In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar
from dataclasses import dataclass

mpl.rcParams.update({
    "text.usetex": False,       # keep sizes stable
    "mathtext.fontset": "cm",   # Computer Modern for math only
    # keep normal font choice as-is (optional)
    # "font.family": "sans-serif",
    # "font.sans-serif": ["DejaVu Sans"],
})

## Fig 1. No feedback baseline (fixed game)

### Baseline game with no environmental feedback

$$M = \begin{bmatrix}
0 & \beta \\
\alpha & 0
\end{bmatrix}
$$

\begin{eqnarray}
\dot{x} = x(1-x)(-(\alpha+\beta)x+\alpha)
\end{eqnarray}

#### Panel A. Stability diagram in the payoff-parameter space ($\alpha, \beta$)

In [ ]:
# phase diagram
def classify(alpha, beta):
    """Output:
       0 -> strategy 1 dominates (x=0 stable)
       1 -> strategy 2 dominates (x=1 stable)
       2 -> coexistence (interior stable)
       3 -> coordination/bistability (0 and 1 stable)
      -1 -> on axes (alpha=0 or beta=0)
    """
    if alpha < 0 and beta > 0:
        return 0
    if alpha > 0 and beta < 0:
        return 1
    if alpha > 0 and beta > 0:
        return 2
    if alpha < 0 and beta < 0:
        return 3
    return -1

A = np.linspace(-1, 1, 201)
B = np.linspace(-1, 1, 201)
AA, BB = np.meshgrid(A, B)
Z = np.zeros_like(AA, dtype=float)
for i in range(AA.shape[0]):
    for j in range(AA.shape[1]):
        Z[i, j] = classify(AA[i, j], BB[i, j])

In [ ]:
COL_HOST    = "#707071"   # host dominance
COL_COEX    = "#629FB7"   # stable coexistence
COL_BISTAB  = "#106E91"   # bistability
COL_CANCER  = "#AF4772"   # cancer dominance

# 0 host dom, 1 cancer dom, 2 coexist, 3 bistability
cmap = ListedColormap([COL_HOST, COL_CANCER, COL_COEX, COL_BISTAB])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

Z_masked = Z.astype(float)
Z_masked[Z_masked < 0] = np.nan  # hide alpha=0 or beta=0 cells

fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(
    Z_masked,
    origin="lower",
    extent=[A.min(), A.max(), B.min(), B.max()],
    aspect="equal",
    cmap=cmap,
    norm=norm,
    interpolation="nearest"
)

ax.axhline(0.001, linewidth=2, color='black')
ax.axvline(0, linewidth=2, color='black')
ax.set(xticks=np.arange(-1, 1.1, 0.5),
       yticks=np.arange(-1, 1.1, 0.5)
      )
ax.set_title(r'Stability regimes in $(\alpha,\beta)$',fontsize=22)
ax.set_xlabel(r'$\alpha$', fontsize=30)
ax.set_ylabel(r'$\beta$', fontsize=30)
ax.tick_params(axis='both', labelsize=14)

# text
txt_kw = dict(
    fontsize=22, color="white", ha="center", va="center", fontweight="semibold",
    path_effects=[pe.withStroke(linewidth=1.2, foreground=(0,0,0,0.6))]
)

ax.text(-0.5,  0.5, "Host\ndominance", **txt_kw)
ax.text( 0.5,  0.5, "Stable\ncoexistence", **txt_kw)
ax.text(-0.5, -0.5, "Bistability", **txt_kw)
ax.text( 0.5, -0.5, "Cancer\ndominance", **txt_kw)

plt.tight_layout()
#plt.savefig('Figures/fig1A_phase_alpha_beta.png', dpi=600, bbox_inches="tight")
#plt.savefig('Figures/fig1A_phase_alpha_beta.svg', bbox_inches="tight")
plt.show()

## Fig 2. $x(t)$ trajectories

In [ ]:
def rhs(t, x, alpha, beta):
    x = x[0]
    dx = x*(1-x)*(alpha - (alpha+beta)*x)
    return [dx]

def sigammalate(alpha, beta, x0, tmax=60, npts=800):
    t_eval = np.linspace(0, tmax, npts)
    sol = solve_ivp(rhs, (0, tmax), [x0], t_eval=t_eval,
                    args=(alpha, beta), rtol=1e-8, atol=1e-10)
    return sol.t, sol.y[0]

def x_star(alpha, beta):
    if alpha*beta > 0 and (alpha+beta) != 0:
        xs = alpha/(alpha+beta)
        if 0 < xs < 1:
            return xs
    return None

In [ ]:
COL = {
    "host":    "#707071",
    "coexist": "#629FB7",
    "bistab":  "#106E91",
    "cancer":  "#AF4772",
}

# Representative parameters (one per quadrant)
cases = [
    ("Host dominance",      -0.5, +0.5, COL["host"]),
    ("Stable coexistence",  +0.5, +0.5, COL["coexist"]),
    ("Bistability",         -0.5, -0.5, COL["bistab"]),
    ("Cancer dominance",    +0.5, -0.5, COL["cancer"]),
]
# Initial conditions (several to show convergence)
x0s_general = [0.02, 0.10, 0.80, 0.98]
# show both basins + "just below/above" the unstable point (~0.5 here)
x0s_bistab  = [0.02, 0.20, 0.49, 0.51, 0.80, 0.98]

# Make Panel B square overall, so each small panel can be square and still large
fig, axs = plt.subplots(2, 2, figsize=(8.2, 8.2), sharex=True, sharey=True, constrained_layout=True)
axs = axs.ravel()

for ax, (name, a, b, color) in zip(axs, cases):
    x0s = x0s_bistab if "Bistability" in name else x0s_general

    for x0 in x0s:
        t, x = sigammalate(a, b, x0, tmax=20)
        ax.plot(t, x, lw=2.8, color=color, alpha=0.95)

    # equilibrium guides (keep subtle)
    ax.axhline(0, ls="--", lw=2, color="black", alpha=0.6)
    ax.axhline(1, ls="--", lw=2, color="black", alpha=0.6)
    xs = x_star(a, b)
    if xs is not None:
        ax.axhline(xs, ls="--", lw=2, color="black", alpha=0.6)

    ax.set_title(name, fontsize=26, pad=6)
    ax.set_xlim(0, 20)
    ax.set_ylim(-0.03, 1.03)

    # square subplot boxes
    #ax.set_box_aspect(1)
    # square subplot boxes (new matplotlib if available; fallback otherwise)
    if hasattr(ax, "set_box_aspect"):
        ax.set_box_aspect(1)
    else:
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        ax.set_aspect(abs((x1 - x0) / (y1 - y0)), adjustable='box')
    # integer-ish ticks on time, clean y ticks
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(labelsize=12)

# axis labels (outer only)
axs[2].set_xlabel("Time", fontsize=22)
axs[3].set_xlabel("Time", fontsize=22)
axs[0].set_ylabel(r"Cancer fraction $x$", fontsize=24)
axs[2].set_ylabel(r"Cancer fraction $x$", fontsize=24)

#plt.savefig("Figures/fig1B_trajectories.svg", bbox_inches="tight")
#plt.savefig("Figures/fig1B_trajectories.png", dpi=600, bbox_inches="tight")

plt.show()


## Fig 4. Environmental feedback trajectories

In [ ]:
# ----------------------------
# Core model building blocks
# ----------------------------

@dataclass(frozen=True)
class GameEndpoints:
    """
    Reduced 2x2 payoff form:
        M = [[0, beta],
             [alpha, 0]]

    Endpoints:
      (alpha_h, beta_h) = baseline (low conditioning)
      (alpha_s, beta_s) = stressed (high conditioning)
    """
    alpha_h: float
    beta_h: float
    alpha_s: float
    beta_s: float


def endpoints_host_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 1 mapping:
      baseline: host-dominant  (alpha<0, beta>0)
      stressed: cancer-dominant (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=-alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


def endpoints_coexist_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 2 mapping:
      baseline: stable coexistence (alpha>0, beta>0)
      stressed: cancer-dominant    (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=+alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


def hill_gate(n, k: float, nu: float = 1.0):
    """
    f(n) = n^nu / (n^nu + k^nu) in [0,1]
    Vectorized.
    """
    n = np.asarray(n)
    n = np.maximum(n, 0.0)
    return (n**nu) / (n**nu + k**nu)


def alpha_beta(n, ends: GameEndpoints, k: float, nu: float = 1.0):
    """
    alpha(n) and beta(n) via interpolation:
      alpha(n) = (1-f) alpha_h + f alpha_s
      beta(n)  = (1-f) beta_h  + f beta_s
    Returns alpha, beta, f (all vectorized).
    """
    f = hill_gate(n, k, nu)
    alpha_n = (1 - f) * ends.alpha_h + f * ends.alpha_s
    beta_n  = (1 - f) * ends.beta_h  + f * ends.beta_s
    return alpha_n, beta_n, f


def rhs(t, y, ends: GameEndpoints, phi: float, gamma: float, k: float, nu: float):
    """
    y = [x, n]
      xdot = x(1-x)( alpha(n) - (alpha(n)+beta(n)) x )
      ndot = phi*x - gamma*n
    """
    x, n = y

    # keep numerics sane
    x = np.clip(x, 0.0, 1.0)
    n = max(n, 0.0)

    a, b, _ = alpha_beta(n, ends, k, nu)
    dx = x * (1 - x) * (a - (a + b) * x)
    dn = phi * x - gamma * n
    return [dx, dn]


def simulate_case(ends: GameEndpoints,
                  phi: float, gamma: float, k: float, nu: float,
                  x0: float, n0: float,
                  tmax: float = 80.0, npts: int = 2000):
    """
    Returns a dict with:
      t, x, n, alpha(t), beta(t), f(t)
    """
    t_eval = np.linspace(0, tmax, npts)
    sol = solve_ivp(
        rhs, (0, tmax), [x0, n0], t_eval=t_eval,
        args=(ends, phi, gamma, k, nu),
        rtol=1e-8, atol=1e-10
    )

    t = sol.t
    x = sol.y[0]
    n = sol.y[1]
    a, b, f = alpha_beta(n, ends, k, nu)

    return {"t": t, "x": x, "n": n, "alpha": a, "beta": b, "f": f}

# ----------------------------
# 4 default cases
# ----------------------------

@dataclass(frozen=True)
class Case:
    name: str
    ends: GameEndpoints
    phi: float
    gamma: float
    k: float
    nu: float
    tmax: float
    x0s: tuple
    n0: float


def build_default_cases():
    cases = {}

    # 1) Clearance (Scenario 1, rho_hat<1)
    cases["clearance"] = Case(
        name="Clearance",
        ends=endpoints_host_to_cancer(alpha=0.25, beta=0.5),
        phi=0.7, gamma=0.5, k=2.0, nu=1.0,
        tmax=80.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    # 2) Stable dormancy / persistence (Scenario 2, rho_hat<1)
    cases["dormancy"] = Case(
        name="Stable dormancy",
        ends=endpoints_coexist_to_cancer(alpha=0.5, beta=0.5),
        phi=0.7, gamma=1.0, k=2.0, nu=1.0,
        tmax=50.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    # 3) Awakening (Scenario 2, rho_hat>1)
    cases["awakening"] = Case(
        name="Awakening",
        ends=endpoints_coexist_to_cancer(alpha=0.1, beta=0.9),
        phi=0.09, gamma=0.02, k=2.0, nu=1.0,
        tmax=720.0,
        x0s=(0.02, 0.10, 0.40, 0.80),
        n0=0.0
    )

    # 4) Primed niche / invasion threshold (Scenario 1, rho_hat>1)
    # Show two initial conditions (notebook: x0=0.2 vs 0.4)
    cases["primed_threshold"] = Case(
        name="Primed niche threshold",
        ends=endpoints_host_to_cancer(alpha=0.5, beta=0.6),
        phi=0.23, gamma=0.5, k=0.1, nu=1.0,
        tmax=50.0,
        x0s=(0.20, 0.40),
        n0=0.1
    )

    return cases

In [ ]:
# Match Fig1 palette
COL = {
    "host":    "#707071",
    "coexist": "#629FB7",
    "bistab":  "#106E91",
    "cancer":  "#AF4772",
}

# Choose one color per feedback case (feel free to change)
CASE_COLOR = {
    "clearance":        COL["host"],
    "dormancy":         COL["coexist"],
    "awakening":        COL["cancer"],
    "primed_threshold": COL["bistab"],
}

# Style knobs to match Fig1
LINE_W = 2.8
LINE_A = 0.95
DASH_W = 2.0
DASH_A = 0.6
TITLE_FS = 20
LABEL_FS = 20
XLABEL_FS = 18
TICK_FS = 12


def run_case(case, npts=2000):
    """Sigammalate all x0s for a Case. Returns list of dict outputs."""
    outs = []
    for x0 in case.x0s:
        out = simulate_case(
            case.ends,
            phi=case.phi, gamma=case.gamma, k=case.k, nu=case.nu,
            x0=x0, n0=case.n0,
            tmax=case.tmax, npts=npts
        )
        out["x0"] = x0
        outs.append(out)
    return outs


def _nice_ylim_n(case, outs):
    """Choose n-axis limits that always show 0 and the tolerance k line."""
    nmax = max(float(np.max(o["n"])) for o in outs)
    top = 1.05 * max(nmax, case.k)
    bottom = -0.03 * case.k  # tiny padding like x axis
    return bottom, top


def plot_case_timeseries(case_key, case, outs, outdir="Figures",
                         layout="stack", label_x0=False):
    """
    Save ONE figure for ONE case.
    layout:
      - "stack": x(t) top, n(t) bottom
      - "twin" : x(t) left axis, n(t) right axis (same panel)
    """
    os.makedirs(outdir, exist_ok=True)
    color = CASE_COLOR.get(case_key, "black")

    if layout == "stack":
        fig, (ax_x, ax_n) = plt.subplots(
            2, 1, figsize=(6.2, 6.2), sharex=True, constrained_layout=True
        )

        # --- x(t) ---
        for o in outs:
            ax_x.plot(o["t"], o["x"], lw=LINE_W, color=color, alpha=LINE_A)
            if label_x0:
                ax_x.text(o["t"][int(0.03*len(o["t"]))], o["x"][int(0.03*len(o["x"]))],
                          f"{o['x0']:.2f}", fontsize=10, color=color)

        ax_x.axhline(0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.axhline(1, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.set_ylim(-0.03, 1.03)
        ax_x.set_title(case.name, fontsize=TITLE_FS, pad=6)
        ax_x.set_ylabel(r"Cancer fraction $x$", fontsize=LABEL_FS)

        # --- n(t) ---
        for o in outs:
            ax_n.plot(o["t"], o["n"], lw=LINE_W, color=color, alpha=LINE_A)

        # tolerance line at n = k
        ax_n.axhline(case.k, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_n.set_ylabel(r"Conditioning $n$", fontsize=LABEL_FS)
        ax_n.set_xlabel("Time", fontsize=XLABEL_FS)
        nlo, nhi = _nice_ylim_n(case, outs)
        ax_n.set_ylim(nlo, nhi)

        # ticks (integer time, clean y)
        ax_n.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax_x.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_n.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_x.tick_params(labelsize=TICK_FS)
        ax_n.tick_params(labelsize=TICK_FS)
        fig.align_ylabels([ax_x, ax_n])

        # Save
        base = os.path.join(outdir, f"fig3_{case_key}_{layout}")
        fig.savefig(base + ".svg", bbox_inches="tight")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight")
        plt.show()
        return base

    elif layout == "twin":
        fig, ax_x = plt.subplots(figsize=(6.2, 6.2), constrained_layout=True)
        ax_n = ax_x.twinx()

        # x(t)
        for o in outs:
            ax_x.plot(o["t"], o["x"], lw=LINE_W, color=color, alpha=LINE_A)
        ax_x.axhline(0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.axhline(1, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.set_ylim(-0.03, 1.03)

        # n(t)
        for o in outs:
            ax_n.plot(o["t"], o["n"], lw=LINE_W, color="black", alpha=0.35)
        ax_n.axhline(case.k, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        nlo, nhi = _nice_ylim_n(case, outs)
        ax_n.set_ylim(nlo, nhi)

        ax_x.set_title(case.name, fontsize=TITLE_FS, pad=6)
        ax_x.set_xlabel("Time", fontsize=XLABEL_FS)
        ax_x.set_ylabel(r"Cancer fraction $x$", fontsize=LABEL_FS)
        ax_n.set_ylabel(r"Conditioning $n$", fontsize=LABEL_FS)

        ax_x.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax_x.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_n.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_x.tick_params(labelsize=TICK_FS)
        ax_n.tick_params(labelsize=TICK_FS)
        fig.align_ylabels([ax_x, ax_n])

        base = os.path.join(outdir, f"fig3_{case_key}_{layout}")
        fig.savefig(base + ".svg", bbox_inches="tight")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight")
        plt.show()
        return base

    else:
        raise ValueError("layout gammast be 'stack' or 'twin'")

In [ ]:
cases = build_default_cases()
all_outs = {}
# one-by-one (recommended)
for key in ["clearance", "dormancy", "awakening", "primed_threshold"]:
    case = cases[key]
    outs = run_case(case, npts=2000)
    all_outs[key] = outs
    # stacked plots (x above, n below)
    # layout = stack OR twin
    plot_case_timeseries(key, case, outs, outdir="Figures", layout="stack",
                         label_x0=(key == "primed_threshold"))

    # If twin-axis versions too:
    #plot_case_timeseries("awakening", cases["awakening"], run_case(cases["awakening"]),
    #                  outdir="Figures", layout="twin")


## Checking $\alpha$ and $\beta$ timeseries

In [ ]:
keys = ['clearance', 'dormancy', 'awakening', 'primed_threshold']
for key in keys:
    print(f"Key = {key}")
    for case in all_outs[key]:
        print(f"x0={case['x0']}")
        plt.plot(case['t'],case['alpha'], label='alpha')
        plt.plot(case['t'],case['beta'], label='beta')
        plt.title('Alpha & Beta',fontsize=18)
        plt.legend()
        plt.show()
        plt.plot(case['t'],case['alpha']/(case['alpha']+case['beta']))
        plt.title('r',fontsize=18)
        plt.show()

## Fig 4. (Nondimensional, reparametrized) Environmental feedback trajectories

In [ ]:
# ----------------------------
# Core model building blocks (NON-DIMENSIONAL STATE; INTEGRATE IN t)
#   state: (x, n̄) with n̄ = n/k
#   parameter: φ̄ = φ/(γ k)
#   integrate in dimensional t
#   option to plot either t or τ=γt
# ----------------------------

@dataclass(frozen=True)
class GameEndpoints:
    """
    Reduced 2x2 payoff-difference form:
        M = [[0, beta],
             [alpha, 0]]

    Endpoints:
      (alpha_h, beta_h) = baseline (low conditioning)
      (alpha_s, beta_s) = stressed  (high conditioning)
    """
    alpha_h: float
    beta_h: float
    alpha_s: float
    beta_s: float


def endpoints_host_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 1 mapping:
      baseline: host-dominant   (alpha<0, beta>0)
      stressed: cancer-dominant (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=-alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


def endpoints_coexist_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 2 mapping:
      baseline: stable coexistence (alpha>0, beta>0)
      stressed: cancer-dominant    (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=+alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


def hill_gate_bar(n_bar, nu: float = 1.0):
    """
    Non-dimensional Hill gate:
        f(n̄) = n̄^nu / (1 + n̄^nu)   in [0,1]
    Vectorized.
    """
    n_bar = np.asarray(n_bar)
    n_bar = np.maximum(n_bar, 0.0)
    return (n_bar**nu) / (1.0 + n_bar**nu)


def alpha_beta_bar(n_bar, ends: GameEndpoints, nu: float = 1.0):
    """
    alpha(n̄) and beta(n̄) via interpolation:
      alpha(n̄) = (1-f) alpha_h + f alpha_s
      beta(n̄)  = (1-f) beta_h  + f beta_s
    Returns alpha, beta, f (vectorized).
    """
    f = hill_gate_bar(n_bar, nu)
    alpha_n = (1 - f) * ends.alpha_h + f * ends.alpha_s
    beta_n  = (1 - f) * ends.beta_h  + f * ends.beta_s
    return alpha_n, beta_n, f


def rhs_t(t, y_bar, ends: GameEndpoints, phi_bar: float, gamma: float, nu: float):
    """
    Integrate in dimensional time t, but with non-dimensional conditioning state n̄=n/k.

      dx/dt     = x(1-x)[ alpha(n̄) - (alpha(n̄)+beta(n̄)) x ]
      dn̄/dt    = γ( φ̄ x - n̄ )     with φ̄ = φ/(γk)

    y_bar = [x, n̄]
    """
    x, n_bar = y_bar
    x = float(np.clip(x, 0.0, 1.0))
    n_bar = float(max(n_bar, 0.0))

    a, b, _ = alpha_beta_bar(n_bar, ends, nu)
    a = float(a); b = float(b)

    dx = x * (1 - x) * (a - (a + b) * x)
    dnbar = gamma * (phi_bar * x - n_bar)
    return [dx, dnbar]


def simulate_case_nondim(ends: GameEndpoints,
                         phi: float, gamma: float, k: float, nu: float,
                         x0: float, n0: float,
                         tmax: float = 80.0, npts: int = 2000):
    """
    Integrate the equations in dimensional time t, using n̄=n/k internally.
    Return both t and τ=γt, and both n̄ and n.

    Returns dict with:
      t, tau, x, nbar, n,
      alpha(t), beta(t), f(t),
      s(t)=alpha+beta, r(t)=alpha/s (NaN if s≈0),
      Lambda(t)=s/gamma, phi_bar
    """
    phi_bar = phi / (gamma * k)
    n0_bar = n0 / k

    t_eval = np.linspace(0.0, tmax, npts)

    sol = solve_ivp(
        rhs_t, (0.0, tmax), [x0, n0_bar],
        t_eval=t_eval,
        args=(ends, phi_bar, gamma, nu),
        rtol=1e-8, atol=1e-10
    )

    t = sol.t
    x = sol.y[0]
    nbar = sol.y[1]

    tau = gamma * t
    n = k * nbar

    a, b, f = alpha_beta_bar(nbar, ends, nu)
    s = a + b

    r = np.full_like(s, np.nan, dtype=float)
    mask = np.abs(s) > 1e-12
    r[mask] = a[mask] / s[mask]

    Lambda = s / gamma  # instantaneous selection/clearance ratio

    return {
        "t": t, "tau": tau,
        "x": x, "nbar": nbar, "n": n,
        "alpha": a, "beta": b, "f": f,
        "s": s, "r": r, "Lambda": Lambda,
        "phi_bar": phi_bar
    }


# ----------------------------
# 4 default cases (same API; phi_bar and nbar computed internally)
# ----------------------------

@dataclass(frozen=True)
class Case:
    name: str
    ends: GameEndpoints
    phi: float
    gamma: float
    k: float
    nu: float
    tmax: float
    x0s: tuple
    n0: float


def build_default_cases():
    cases = {}

    # 1) Clearance (Scenario 1, phi_bar < 1)
    cases["clearance"] = Case(
        name="Clearance",
        ends=endpoints_host_to_cancer(alpha=0.25, beta=0.5),
        phi=0.7, gamma=0.5, k=2.0, nu=1.0,
        tmax=80.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    # 2) Stable dormancy / persistence (Scenario 2, phi_bar < 1)
    cases["dormancy"] = Case(
        name="Stable dormancy",
        ends=endpoints_coexist_to_cancer(alpha=0.5, beta=0.5),
        phi=0.7, gamma=1.0, k=2.0, nu=1.0,
        tmax=50.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    # 3) Awakening (Scenario 2, phi_bar > 1)
    cases["awakening"] = Case(
        name="Awakening",
        ends=endpoints_coexist_to_cancer(alpha=0.1, beta=0.9),
        phi=0.09, gamma=0.02, k=2.0, nu=1.0,
        tmax=720.0,
        x0s=(0.02, 0.10, 0.40, 0.80),
        n0=0.0
    )

    # 4) Primed niche / invasion threshold (Scenario 1, phi_bar > 1)
    cases["primed_threshold"] = Case(
        name="Primed niche threshold",
        ends=endpoints_host_to_cancer(alpha=0.5, beta=0.6),
        phi=0.23, gamma=0.5, k=0.1, nu=1.0,
        tmax=50.0,
        x0s=(0.20, 0.40),
        n0=0.1
    )

    return cases


# ----------------------------
# Plot styling (match Fig1)
# ----------------------------

COL = {
    "host":    "#707071",
    "coexist": "#629FB7",
    "bistab":  "#106E91",
    "cancer":  "#AF4772",
}

CASE_COLOR = {
    "clearance":        COL["host"],
    "dormancy":         COL["coexist"],
    "awakening":        COL["cancer"],
    "primed_threshold": COL["bistab"],
}

LINE_W = 2.8
LINE_A = 0.95
DASH_W = 2.0
DASH_A = 0.6
TITLE_FS = 26
LABEL_FS = 22#20
XLABEL_FS = 26#18
TICK_FS = 12


def run_case(case: Case, npts=2000):
    """Simulate all x0s for a Case. Returns list of dict outputs."""
    outs = []
    for x0 in case.x0s:
        out = simulate_case_nondim(
            case.ends,
            phi=case.phi, gamma=case.gamma, k=case.k, nu=case.nu,
            x0=x0, n0=case.n0,
            tmax=case.tmax, npts=npts
        )
        out["x0"] = x0
        outs.append(out)
    return outs


def _nice_ylim_nbar(outs):
    """n̄-axis limits; always include 0 and show tolerance line at n̄=1."""
    nbar_max = max(float(np.max(o["nbar"])) for o in outs)
    top = 1.05 * max(nbar_max, 1.0)
    bottom = -0.03
    return bottom, top


def plot_case_timeseries(case_key, case: Case, outs, outdir="Figures",
                         layout="stack", time_axis="t", label_x0=False):
    """
    Save ONE figure for ONE case.

    layout:
      - "stack": x(top), n̄(bottom)
      - "twin" : x left axis, n̄ right axis (same panel)

    time_axis:
      - "t"   : dimensional time t
      - "tau" : scaled time τ = γ t
    """
    os.makedirs(outdir, exist_ok=True)
    color = CASE_COLOR.get(case_key, "black")

    def _T(o):
        return o["t"] if time_axis == "t" else o["tau"]

    def _curve_color(o):
        if case_key == "primed_threshold":
            # classify by final state (robust + zero extra bookkeeping)
            return COL["cancer"] if o["x"][-1] > 0.5 else COL["host"]
        return CASE_COLOR.get(case_key, "black")
    
    xlab = "Time" if time_axis == "t" else "Time"

    if layout == "stack":
        fig, (ax_x, ax_n) = plt.subplots(
            2, 1, figsize=(6.2, 6.2), sharex=True, constrained_layout=True
        )

        # --- x(t) ---
        for o in outs:
            c = _curve_color(o)
            ax_x.plot(_T(o), o["x"], lw=LINE_W, color=c, alpha=LINE_A)

        ax_x.axhline(0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.axhline(1, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.set_ylim(-0.03, 1.03)
        ax_x.set_title(case.name, fontsize=TITLE_FS, pad=6)
        ax_x.set_ylabel(r"Cancer fraction $x$", fontsize=LABEL_FS)

        # --- n̄(t) ---
        for o in outs:
            c = _curve_color(o)
            ax_n.plot(_T(o), o["nbar"], lw=LINE_W, color=c, alpha=LINE_A)
            if label_x0:
                i = int(0.03 * len(_T(o)))
                ax_n.text(_T(o)[i], o["nbar"][i], f"{o['x0']:.2f}",
                          fontsize=10, color=c)

        ax_n.axhline(1.0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)  # tolerance at n̄=1
        ax_n.set_ylabel(r"Conditioning $\bar n$", fontsize=LABEL_FS)
        ax_n.set_xlabel(xlab, fontsize=XLABEL_FS)
        nlo, nhi = _nice_ylim_nbar(outs)
        ax_n.set_ylim(nlo, nhi)

        # ticks: integer-ish on x; fixed-format y to keep label widths aligned
        ax_n.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax_x.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_n.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_x.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax_n.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax_x.tick_params(labelsize=TICK_FS)
        ax_n.tick_params(labelsize=TICK_FS)
        fig.align_ylabels([ax_x, ax_n])

        base = os.path.join(outdir, f"fig3_{case_key}_{layout}_{time_axis}")
        fig.savefig(base + ".svg", bbox_inches="tight")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight")
        plt.show()
        return base

    elif layout == "twin":
        fig, ax_x = plt.subplots(figsize=(6.2, 6.2), constrained_layout=True)
        ax_n = ax_x.twinx()

        for o in outs:
            c = _curve_color(o)
            ax_x.plot(_T(o), o["x"], lw=LINE_W, color=c, alpha=LINE_A)
        ax_x.axhline(0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.axhline(1, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        ax_x.set_ylim(-0.03, 1.03)

        for o in outs:
            c = _curve_color(o)
            ax_n.plot(_T(o), o["nbar"], lw=LINE_W, color="black", alpha=0.30)
        ax_n.axhline(1.0, ls="--", lw=DASH_W, color="black", alpha=DASH_A)
        nlo, nhi = _nice_ylim_nbar(outs)
        ax_n.set_ylim(nlo, nhi)

        ax_x.set_title(case.name, fontsize=TITLE_FS, pad=6)
        ax_x.set_xlabel(xlab, fontsize=XLABEL_FS)
        ax_x.set_ylabel(r"Cancer fraction $x$", fontsize=LABEL_FS)
        ax_n.set_ylabel(r"Conditioning $\bar n$", fontsize=LABEL_FS)

        ax_x.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        ax_x.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_n.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_x.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax_n.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax_x.tick_params(labelsize=TICK_FS)
        ax_n.tick_params(labelsize=TICK_FS)
        fig.align_ylabels([ax_x, ax_n])

        base = os.path.join(outdir, f"fig3_{case_key}_{layout}_{time_axis}")
        fig.savefig(base + ".svg", bbox_inches="tight")
        fig.savefig(base + ".png", dpi=600, bbox_inches="tight")
        plt.show()
        return base

    else:
        raise ValueError("layout must be 'stack' or 'twin'.")


# -------------------------
# Example usage (one-by-one)
# -------------------------

cases = build_default_cases()

for key in ["clearance", "dormancy", "awakening", "primed_threshold"]:
    case = cases[key]
    outs = run_case(case, npts=2000)

    # Integrate in t, but choose what to plot on the x-axis here:
    plot_case_timeseries(
        key, case, outs,
        outdir="Figures",
        layout="stack",
        time_axis="tau",      # "t" or "tau"
        label_x0=(key == "primed_threshold")
    )


### Fig 5. Phase portraits

In [ ]:
# ==========================================================
# Figure 5 — Phase portraits (2x2 + individual panels)
# Definitive, self-contained script:
#   - Removes nbar=1 line
#   - No panel letters A/B/C/D
#   - Saves 2x2 with ONE shared legend
#   - Saves each panel separately (no legend)
#   - Adds plot padding so fixed points/trajectories aren't clipped
#   - Uses add_arrows_to_line2D on Line2D trajectories
#   - Fixes spurious interior equilibrium/saddle by solving a continuous equation
# ==========================================================

In [ ]:
# ----------------------------
# Colors (match Fig1 palette)
# ----------------------------
COL = {
    "host":    "#707071",
    "coexist": "#629FB7",
    "bistab":  "#106E91",
    "cancer":  "#AF4772",
}

CASE_COLOR = {
    "clearance":        COL["host"],
    "dormancy":         COL["coexist"],
    "awakening":        COL["cancer"],
    "primed_threshold": COL["bistab"],
}


# ----------------------------
# Core model (non-dimensional state; integrate in tau = gamma t)
# ----------------------------
@dataclass(frozen=True)
class GameEndpoints:
    """
    Reduced payoff-difference form:
        M = [[0, beta],
             [alpha, 0]]

    Endpoints:
      (alpha_h, beta_h) = baseline (low conditioning)
      (alpha_s, beta_s) = stressed  (high conditioning)
    """
    alpha_h: float
    beta_h: float
    alpha_s: float
    beta_s: float


def endpoints_host_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 1 mapping:
      baseline: host-dominant   (alpha<0, beta>0)
      stressed: cancer-dominant (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=-alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


def endpoints_coexist_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """
    Scenario 2 mapping:
      baseline: stable coexistence (alpha>0, beta>0)
      stressed: cancer-dominant    (alpha>0, beta<0)
    """
    return GameEndpoints(alpha_h=+alpha, beta_h=+beta,
                         alpha_s=+alpha, beta_s=-beta)


@dataclass(frozen=True)
class Case:
    name: str
    ends: GameEndpoints
    phi: float
    gamma: float
    k: float
    nu: float
    tmax: float
    x0s: tuple
    n0: float


def build_default_cases():
    cases = {}

    cases["clearance"] = Case(
        name="Clearance",
        ends=endpoints_host_to_cancer(alpha=0.25, beta=0.5),
        phi=0.7, gamma=0.5, k=2.0, nu=1.0,
        tmax=80.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    cases["dormancy"] = Case(
        name="Stable dormancy",
        ends=endpoints_coexist_to_cancer(alpha=0.5, beta=0.5),
        phi=0.7, gamma=1.0, k=2.0, nu=1.0,
        tmax=50.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0
    )

    cases["awakening"] = Case(
        name="Awakening",
        ends=endpoints_coexist_to_cancer(alpha=0.1, beta=0.9),
        phi=0.09, gamma=0.02, k=2.0, nu=1.0,
        tmax=720.0,
        x0s=(0.02, 0.10, 0.40, 0.80),
        n0=0.0
    )

    cases["primed_threshold"] = Case(
        name="Primed niche threshold",
        ends=endpoints_host_to_cancer(alpha=0.5, beta=0.6),
        phi=0.23, gamma=0.5, k=0.1, nu=1.0,
        tmax=50.0,
        x0s=(0.20, 0.40),
        n0=0.1
    )

    return cases


def phi_bar(case: Case) -> float:
    """phi_bar = phi/(gamma*k)"""
    return case.phi / (case.gamma * case.k)


def hill_gate_bar(n_bar, nu: float = 1.0):
    """f(n̄) = n̄^nu / (1 + n̄^nu), vectorized, n̄>=0."""
    n_bar = np.asarray(n_bar)
    n_bar = np.maximum(n_bar, 0.0)
    return (n_bar**nu) / (1.0 + n_bar**nu)


def alpha_beta_bar(n_bar, ends: GameEndpoints, nu: float = 1.0):
    """Linear interpolation between endpoints via f(n̄)."""
    f = hill_gate_bar(n_bar, nu)
    alpha_n = (1 - f) * ends.alpha_h + f * ends.alpha_s
    beta_n  = (1 - f) * ends.beta_h  + f * ends.beta_s
    return alpha_n, beta_n, f


def rhs_tau(_tau, y, ends: GameEndpoints, phi_bar_val: float, gamma: float, nu: float):
    """
    tau = gamma t
      dx/dtau    = (1/gamma) x(1-x)[ a(nbar) - (a(nbar)+b(nbar)) x ]
      dnbar/dtau = phi_bar x - nbar
    """
    x, nbar = y
    x = float(np.clip(x, 0.0, 1.0))
    nbar = float(max(nbar, 0.0))

    a, b, _ = alpha_beta_bar(nbar, ends, nu)
    a = float(a); b = float(b)

    dx = (1.0 / gamma) * x * (1 - x) * (a - (a + b) * x)
    dn = phi_bar_val * x - nbar
    return [dx, dn]


def integrate_traj_tau(case: Case, x0: float, nbar0: float, tau_max=None, npts=1200):
    """Integrate one trajectory in tau and return arrays (tau, x(tau), nbar(tau))."""
    if tau_max is None:
        tau_max = case.gamma * case.tmax

    pb = phi_bar(case)
    tau_eval = np.linspace(0.0, tau_max, npts)

    sol = solve_ivp(
        rhs_tau, (0.0, tau_max), [x0, nbar0],
        t_eval=tau_eval,
        args=(case.ends, pb, case.gamma, case.nu),
        rtol=1e-8, atol=1e-10
    )
    return sol.t, sol.y[0], sol.y[1]


def x_nullcline_curve(case: Case, nbar_vals: np.ndarray, eps=1e-10):
    """
    Interior x-nullcline (dx=0):
        x = a(nbar)/(a(nbar)+b(nbar))
    Return x*(nbar) with NaNs where undefined/outside (0,1) or near singular.
    """
    a, b, _ = alpha_beta_bar(nbar_vals, case.ends, case.nu)
    s = a + b

    xstar = np.full_like(nbar_vals, np.nan, dtype=float)
    mask = np.abs(s) > eps
    xstar[mask] = (a[mask] / s[mask]).astype(float)

    xstar[(xstar <= 0.0) | (xstar >= 1.0)] = np.nan
    return xstar


def fprime_hill(nbar: float, nu: float) -> float:
    """Derivative of f(nbar)=nbar^nu/(1+nbar^nu)."""
    nbar = float(max(nbar, 0.0))
    return nu * (nbar ** (nu - 1.0)) / ((1.0 + nbar ** nu) ** 2)


def jacobian_tau(case: Case, x: float, nbar: float) -> np.ndarray:
    """
    Jacobian of tau-system:
      F = dx/dtau = (1/gamma)*x(1-x)[a - (a+b)x]
      G = dnbar/dtau = phi_bar*x - nbar
    """
    pb = phi_bar(case)
    gamma = case.gamma
    nu = case.nu
    ends = case.ends

    a, b, _ = alpha_beta_bar(nbar, ends, nu)
    a = float(a); b = float(b)

    p = x * (1.0 - x)
    q = a - (a + b) * x
    dp = 1.0 - 2.0 * x
    dq = -(a + b)
    Fx = (1.0 / gamma) * (dp * q + p * dq)

    fp = fprime_hill(nbar, nu)
    a_n = fp * (ends.alpha_s - ends.alpha_h)
    b_n = fp * (ends.beta_s  - ends.beta_h)
    s_n = a_n + b_n

    dq_n = a_n - s_n * x
    Fn = (1.0 / gamma) * (p * dq_n)

    Gx = pb
    Gn = -1.0

    return np.array([[Fx, Fn], [Gx, Gn]], dtype=float)


def classify_equilibrium(case: Case, x: float, nbar: float, tol=1e-8) -> str:
    eig = np.linalg.eigvals(jacobian_tau(case, x, nbar))
    re = np.real(eig)
    if np.all(re < -tol):
        return "stable"
    if np.all(re > tol):
        return "unstable"
    if re[0] * re[1] < -tol:
        return "saddle"
    return "neutral"


# ----------------------------
# CRITICAL FIX: interior equilibria without discontinuity
# ----------------------------
def find_interior_equilibria(case: Case):
    """
    Interior equilibria satisfy:
      nbar = pb*x
      and dx=0 interior:  a(nbar) - (a(nbar)+b(nbar))x = 0

    Equivalent continuous equation (no division):
      h(x) = x*(a+b) - a = 0, evaluated at nbar=pb*x.
    """
    pb = phi_bar(case)
    ends = case.ends
    nu = case.nu

    def h(x):
        if not (0.0 < x < 1.0):
            return np.nan
        nbar = pb * x
        a, b, _ = alpha_beta_bar(nbar, ends, nu)
        a = float(a); b = float(b)
        return x * (a + b) - a

    xs = np.linspace(1e-6, 1.0 - 1e-6, 8000)
    hv = np.array([h(x) for x in xs], dtype=float)

    roots = []
    for i in range(len(xs) - 1):
        h1, h2 = hv[i], hv[i + 1]
        if np.isnan(h1) or np.isnan(h2):
            continue
        if h1 == 0.0:
            roots.append(xs[i])
            continue
        if h1 * h2 < 0.0:
            try:
                r = brentq(h, xs[i], xs[i + 1], maxiter=200)
                roots.append(float(r))
            except Exception:
                pass

    if len(roots) == 0:
        res = minimize_scalar(lambda z: abs(h(z)), bounds=(1e-6, 1.0 - 1e-6), method="bounded")
        if res.success and abs(h(res.x)) < 1e-9:
            roots.append(float(res.x))

    roots = sorted(roots)
    uniq = []
    for r in roots:
        if not uniq or abs(r - uniq[-1]) > 1e-5:
            uniq.append(r)

    return [(r, pb * r) for r in uniq]


def compute_equilibria(case: Case):
    pb = phi_bar(case)
    eqs = [
        {"x": 0.0, "nbar": 0.0},
        {"x": 1.0, "nbar": pb},
    ]
    for (xI, nI) in find_interior_equilibria(case):
        eqs.append({"x": xI, "nbar": nI})
    for e in eqs:
        e["kind"] = classify_equilibrium(case, e["x"], e["nbar"])
    return eqs


# ----------------------------
# Arrow helper
# ----------------------------
def add_arrows_to_line2D(line, n_arrows=3, arrowstyle='->', lw=1):
    """
    Add `n_arrows` arrows along a 2D line (Line2D object from plt.plot).
    Arrows are approximately equally spaced by arc length.
    """
    ax = line.axes
    x = line.get_xdata()
    y = line.get_ydata()
    color = line.get_color()

    N = len(x)
    if N < 2:
        return

    dx = np.diff(x)
    dy = np.diff(y)
    ds = np.sqrt(dx**2 + dy**2)
    s = np.concatenate(([0.0], np.cumsum(ds)))

    if s[-1] == 0:
        return

    s_norm = s / s[-1]
    targets = np.linspace(0.2, 0.8, n_arrows)

    for t in targets:
        idx = np.searchsorted(s_norm, t)
        if idx <= 0 or idx >= N:
            continue

        x_start, y_start = x[idx-1], y[idx-1]
        x_end,   y_end   = x[idx],   y[idx]

        ax.annotate(
            "",
            xy=(x_end, y_end), xytext=(x_start, y_start),
            arrowprops=dict(
                arrowstyle=arrowstyle,
                color=color,
                lw=lw,
                shrinkA=0,
                shrinkB=0,
            ),
        )


# ----------------------------
# Basin shading for bistable case (fast Euler)
# ----------------------------
def basin_map_euler(case: Case, ylim, nx=151, ny=151, tau_end=None, dtau=0.02):
    pb = phi_bar(case)
    if tau_end is None:
        tau_end = case.gamma * case.tmax

    xs = np.linspace(0.0, 1.0, nx)
    ys = np.linspace(ylim[0], ylim[1], ny)
    X, Y = np.meshgrid(xs, ys)

    x = X.copy()
    nbar = Y.copy()

    steps = int(tau_end / dtau)
    ends = case.ends
    nu = case.nu
    gamma = case.gamma

    for _ in range(steps):
        a, b, _ = alpha_beta_bar(nbar, ends, nu)
        dx = (1.0 / gamma) * x * (1 - x) * (a - (a + b) * x)
        dn = pb * x - nbar

        x = np.clip(x + dtau * dx, 0.0, 1.0)
        nbar = np.maximum(nbar + dtau * dn, 0.0)

    Z = (x > 0.5).astype(float)  # 1=cancer basin, 0=host basin
    return X, Y, Z


# ----------------------------
# Shared legend handles (NO nbar=1)
# ----------------------------
def common_fig4_legend_handles(include_basin_items=True):
    handles = [
        #Line2D([0],[0], color="black", ls="--", lw=2.0, alpha=0.6,
        #       label=r"$\dot{\bar n}=0$  ($\bar n=\bar\phi x$)"),
        #Line2D([0],[0], color="black", ls="-", lw=2.0, alpha=0.85,
        #       label=r"$\dot x=0$ (interior)"),
        Line2D([0],[0], marker="o", color="none",
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=1.5, markersize=7,
               label="initial condition"),
        Line2D([0],[0], marker="o", color="none",
               markerfacecolor="black", markeredgecolor="white",
               markeredgewidth=1.5, markersize=8,
               label="stable equilibrium"),
        Line2D([0],[0], marker="*", color="none",
               markerfacecolor="black", markeredgecolor="black",
               markersize=12,
               label="saddle"),
        Line2D([0],[0], color="black", lw=2.5, alpha=0.9,
               label="trajectories"),
    ]
    if include_basin_items:
        handles += [
            Line2D([0],[0], color="black", lw=2.2, label="separatrix"),
            Line2D([0],[0], color=COL["host"],   lw=3, alpha=0.8, label="host basin"),
            Line2D([0],[0], color=COL["cancer"], lw=3, alpha=0.8, label="cancer basin"),
        ]
    return handles


# ----------------------------
# Main single-panel plotter
# ----------------------------
def plot_phase_portrait(
    ax,
    case_key: str,
    case: Case,
    *,
    quiver_n=(23, 23),
    nbar_ylim=None,
    n_ic=14,
    seed=1,
    compute_basins_for_bistable=True,
    xpad=0.04,
    ypad_frac=0.06,
):
    pb = phi_bar(case)

    # y-range for computations (>=0)
    if nbar_ylim is None:
        ymax = max(1.2, 1.15 * max(1.0, pb))
        nbar_ylim_data = (0.0, ymax)
    else:
        nbar_ylim_data = (max(0.0, float(nbar_ylim[0])), float(nbar_ylim[1]))

    # y-range for plotting (padding; can dip slightly below 0)
    ylow_plot = -ypad_frac * nbar_ylim_data[1]
    yhigh_plot = nbar_ylim_data[1] * (1.0 + ypad_frac)

    # basins + separatrix only for primed_threshold
    if compute_basins_for_bistable and case_key == "primed_threshold":
        Xg, Yg, Z = basin_map_euler(case, nbar_ylim_data, nx=151, ny=151, dtau=0.02)
        ax.contourf(Xg, Yg, Z, levels=[-0.5, 0.5, 1.5],
                    colors=[COL["host"], COL["cancer"]], alpha=0.12)
        ax.contour(Xg, Yg, Z, levels=[0.5], colors="black", linewidths=2.2, alpha=0.9)

    # vector field (normalized direction)
    xq = np.linspace(0.02, 0.98, quiver_n[0])
    yq = np.linspace(nbar_ylim_data[0], nbar_ylim_data[1], quiver_n[1])
    X, Y = np.meshgrid(xq, yq)

    a, b, _ = alpha_beta_bar(Y, case.ends, case.nu)
    dx = (1.0 / case.gamma) * X * (1 - X) * (a - (a + b) * X)
    dn = pb * X - Y

    speed = np.sqrt(dx * dx + dn * dn)
    dxn = np.where(speed > 0, dx / speed, 0.0)
    dnn = np.where(speed > 0, dn / speed, 0.0)

    ax.quiver(X, Y, dxn, dnn,
              angles="xy", scale_units="xy", scale=35,
              width=0.003, headwidth=3.5, headlength=4.5, headaxislength=4.0,
              color="black", alpha=0.6)

    # nullclines
    """
    xx = np.linspace(0.0, 1.0, 600)
    ax.plot(xx, pb * xx, ls="--", lw=2.0, color="black", alpha=0.6)  # dnbar=0

    nvals = np.linspace(nbar_ylim_data[0], nbar_ylim_data[1], 800)
    xstar = x_nullcline_curve(case, nvals)
    ax.plot(xstar, nvals, lw=2.0, color="black", alpha=0.85)         # dx=0 interior
    """
    # equilibria
    eqs = compute_equilibria(case)
    for e in eqs:
        xeq, yeq, kind = e["x"], e["nbar"], e["kind"]
        if kind == "stable":
            ax.plot(xeq, yeq, marker="o", ms=9, mfc="black", mec="white", mew=1.5, linestyle="None", zorder=7)
        elif kind == "saddle":
            ax.plot(xeq, yeq, marker="*", ms=14, mfc="black", mec="black", linestyle="None", zorder=8)
        elif kind == "unstable":
            ax.plot(xeq, yeq, marker="o", ms=9, mfc="white", mec="black", mew=1.5, linestyle="None", zorder=7)

    # trajectories
    rng = np.random.default_rng(seed)
    xs0 = rng.uniform(0.05, 0.95, size=n_ic)
    ys0 = rng.uniform(nbar_ylim_data[0] + 0.02, nbar_ylim_data[1] - 0.02, size=n_ic)

    # deterministic ICs
    extra_x = np.array([0.1, 0.9, 0.5, 0.2, 0.8])
    extra_y = np.array([0.0, 0.0, min(1.0, nbar_ylim_data[1]-0.02), 0.2,
                        min(nbar_ylim_data[1] * 0.8, nbar_ylim_data[1]-0.02)])
    xs0 = np.concatenate([xs0, extra_x])
    ys0 = np.concatenate([ys0, extra_y])

    tau_max = case.gamma * case.tmax
    base_color = CASE_COLOR.get(case_key, "black")

    for (x0, y0) in zip(xs0, ys0):
        _, xt, nt = integrate_traj_tau(case, float(x0), float(y0), tau_max=tau_max, npts=1200)

        c = base_color
        if case_key == "primed_threshold":
            c = COL["cancer"] if xt[-1] > 0.5 else COL["host"]

        line = ax.plot(xt, nt, lw=2.2, color=c, alpha=0.92, zorder=5)[0]
        add_arrows_to_line2D(line, n_arrows=2, arrowstyle="->", lw=1.6)

        ax.plot(xt[0], nt[0], marker="o", ms=5, mfc="white", mec=c, mew=1.4, linestyle="None", zorder=6)

    # formatting + padding
    ax.set_xlim(-xpad, 1.0 + xpad)
    ax.set_ylim(ylow_plot, yhigh_plot)
    ax.set_title(case.name, fontsize=18, pad=6)
    ax.set_xlabel(r"Cancer fraction $x$", fontsize=14)
    ax.set_ylabel(r"Conditioning $\bar n$", fontsize=14)
    ax.tick_params(labelsize=11)

def enforce_square_axes(axs):
    """Force every axis box to be square."""
    for ax in np.ravel(axs):
        # modern Matplotlib
        if hasattr(ax, "set_box_aspect"):
            ax.set_box_aspect(1)
        else:
            make_axes_box_square(ax)


def make_axes_box_square(ax):
    """
    Force the *axes box* (panel) to be square in figure coordinates.
    Fallback for older matplotlib.
    """
    fig = ax.figure
    fig.canvas.draw()  # finalize positions
    b = ax.get_position()
    s = min(b.width, b.height)
    cx = 0.5 * (b.x0 + b.x1)
    cy = 0.5 * (b.y0 + b.y1)
    ax.set_position([cx - s/2, cy - s/2, s, s])
    
# ----------------------------
# Build the 2x2 figure + individual panels
# ----------------------------
def plot_fig4_phase_portraits(
    cases: dict,
    outdir="Figures",
    filename_base="fig4_phase_portraits",
    include_basin_items=True,
    save_individual=True,
):
    os.makedirs(outdir, exist_ok=True)
    order = ["clearance", "dormancy", "awakening", "primed_threshold"]

    # --- 2x2 with shared legend ---
    # Make the FIGURE square so square panels are actually feasible.
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    for ax, key in zip(axs.ravel(), order):
        plot_phase_portrait(
            ax, key, cases[key],
            n_ic=14,
            seed=1,
            compute_basins_for_bistable=True,
        )

    # Make all panels square AFTER plotting and BEFORE any layout/legend.
    enforce_square_axes(axs)

    # Manual spacing (DO NOT use tight_layout here; it breaks box_aspect)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.22,
                        wspace=0.28, hspace=0.28)

    # Shared legend
    handles = common_fig4_legend_handles(include_basin_items=include_basin_items)
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=4,
        frameon=True,
        fontsize=9,
        bbox_to_anchor=(0.5, 0.07),
    )

    out_svg = os.path.join(outdir, filename_base + ".svg")
    out_png = os.path.join(outdir, filename_base + ".png")
    fig.savefig(out_svg)                 # <-- no bbox_inches="tight" (it can distort layout)
    fig.savefig(out_png, dpi=600)
    plt.close(fig)

    # --- individual panels (no legend) ---
    if save_individual:
        for key in order:
            # Make the SINGLE figure square too
            fig1, ax1 = plt.subplots(figsize=(5.2, 5.2))
            plot_phase_portrait(
                ax1, key, cases[key],
                n_ic=14,
                seed=1,
                compute_basins_for_bistable=True,
            )

            # Force the single axis to be square
            enforce_square_axes([ax1])

            # Manual margins (no tight_layout)
            fig1.subplots_adjust(left=0.16, right=0.98, top=0.90, bottom=0.14)

            fsvg = os.path.join(outdir, f"{filename_base}_{key}.svg")
            fpng = os.path.join(outdir, f"{filename_base}_{key}.png")
            fig1.savefig(fsvg)
            fig1.savefig(fpng, dpi=600)
            plt.close(fig1)

    return out_svg, out_png

In [ ]:
if __name__ == "__main__":
    cases = build_default_cases()
    plot_fig4_phase_portraits(
        cases,
        outdir="Figures",
        filename_base="fig4_phase_portraits",
        include_basin_items=True,
        save_individual=True
    )

## Reviewer comments

### R1M2

Clinically, an important question, especially in the context of breast cancer, is why awakening times are so diverse between patients. Could you explicitly address this with your results? For instance, in the awakening scenario, how variable awakening times are given different initial conditions (varying parameter values too may be future work)? Similarly for the priming scenario. (This is a reframing of the results you already have that would increase its impact, I think.)

In [ ]:
# ============================================================
# Reviewer 1, Major Comment 2
# Time-to-outgrowth and dynamical-threshold analysis
#
# AFTER the final phase-portrait/model-definition cell.
# It uses:
#   Case
#   GameEndpoints
#   build_default_cases
#   phi_bar
#   alpha_beta_bar
#   compute_equilibria
#   jacobian_tau
# ============================================================
# Revision notes implemented here:
#   - No hard-coded manuscript figure number such as 'Fig. 4C'.
#   - The representative awakening parameter is labeled generically,
#     so the code remains valid if figure numbering changes.
#   - Numerical arrays are saved to Figures/latency_analysis_data.npz
#     for reproducibility and optional replotting.
#   - Run this cell/script after the final model-definition and
#     phase-portrait cells in Paper.ipynb.
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from scipy.integrate import solve_ivp


# ------------------------------------------------------------
# Check that the final notebook model definitions are available
# ------------------------------------------------------------

_REQUIRED_NAMES = [
    "Case",
    "build_default_cases",
    "phi_bar",
    "alpha_beta_bar",
    "compute_equilibria",
    "jacobian_tau",
]

_missing = [name for name in _REQUIRED_NAMES if name not in globals()]

if _missing:
    raise RuntimeError(
        "Run the final model/phase-portrait cell before this cell. "
        f"Missing definitions: {_missing}"
    )


# Fall-back colours if this cell is run independently of the plotting cells
if "COL" not in globals():
    COL = {
        "host": "#707071",
        "coexist": "#629FB7",
        "bistab": "#106E91",
        "cancer": "#AF4772",
    }


LATENCY_OUTDIR = "Figures"
os.makedirs(LATENCY_OUTDIR, exist_ok=True)


# ------------------------------------------------------------
# Unclipped tau-system
# ------------------------------------------------------------

def latency_rhs_tau(
    tau,
    y,
    ends,
    phi_bar_val,
    gamma,
    nu=1.0,
):
    """
    Nondimensional-state system integrated in tau = gamma*t:

        dx/dtau = (1/gamma) x(1-x)
                  [alpha(nbar) - (alpha(nbar)+beta(nbar))x]

        dnbar/dtau = phi_bar*x - nbar

    This version does not clip x or nbar inside the RHS, so that
    local stability and first-passage calculations are not altered
    by numerical projection.
    """
    x, nbar = y

    alpha_n, beta_n, _ = alpha_beta_bar(nbar, ends, nu)
    alpha_n = float(alpha_n)
    beta_n = float(beta_n)

    dx = (
        (1.0 / gamma)
        * x
        * (1.0 - x)
        * (alpha_n - (alpha_n + beta_n) * x)
    )

    dnbar = phi_bar_val * x - nbar

    return np.array([dx, dnbar], dtype=float)


# ------------------------------------------------------------
# Construct a case with a specified environmental load
# ------------------------------------------------------------

def case_at_phi_bar(base_case, phi_bar_val):
    """
    Return a copy of base_case with a specified phi_bar while
    keeping gamma and k fixed.

        phi = phi_bar * gamma * k
    """
    return Case(
        name=base_case.name,
        ends=base_case.ends,
        phi=phi_bar_val * base_case.gamma * base_case.k,
        gamma=base_case.gamma,
        k=base_case.k,
        nu=base_case.nu,
        tmax=base_case.tmax,
        x0s=base_case.x0s,
        n0=base_case.n0,
    )


# ------------------------------------------------------------
# Exact first-passage time using solve_ivp
# ------------------------------------------------------------

def time_to_outgrowth(
    case,
    x0,
    nbar0,
    x_out=0.9,
    tau_max=5000.0,
    rtol=1e-9,
    atol=1e-11,
    max_step=1.0,
):
    """
    First upward crossing of x_out:

        T_out = inf{tau > 0 : x(tau) = x_out, dx/dtau > 0}.

    Returns np.nan if no crossing is detected before tau_max.

    Initial conditions must satisfy x0 < x_out.
    """
    if not (0.0 <= x0 < x_out):
        return np.nan

    pb = phi_bar(case)

    def outgrowth_event(tau, y, *args):
        return y[0] - x_out

    outgrowth_event.terminal = True
    outgrowth_event.direction = 1

    sol = solve_ivp(
        latency_rhs_tau,
        (0.0, tau_max),
        [x0, nbar0],
        args=(case.ends, pb, case.gamma, case.nu),
        events=outgrowth_event,
        method="DOP853",
        rtol=rtol,
        atol=atol,
        max_step=max_step,
    )

    if sol.t_events[0].size:
        return float(sol.t_events[0][0])

    return np.nan


# ------------------------------------------------------------
# Vectorized RK4 first-passage solver for heatmaps
# ------------------------------------------------------------

def latency_rhs_tau_array(
    x,
    nbar,
    phi_bar_values,
    ends,
    gamma,
    nu=1.0,
):
    """
    Vectorized form of the tau-system for heatmap calculations.
    """
    nbar_positive = np.maximum(nbar, 0.0)

    f = (
        nbar_positive**nu
        / (1.0 + nbar_positive**nu)
    )

    alpha_n = (
        (1.0 - f) * ends.alpha_h
        + f * ends.alpha_s
    )

    beta_n = (
        (1.0 - f) * ends.beta_h
        + f * ends.beta_s
    )

    dx = (
        (1.0 / gamma)
        * x
        * (1.0 - x)
        * (alpha_n - (alpha_n + beta_n) * x)
    )

    dnbar = phi_bar_values * x - nbar

    return dx, dnbar


def time_to_outgrowth_grid(
    x0,
    nbar0,
    phi_bar_values,
    ends,
    gamma,
    nu=1.0,
    x_out=0.9,
    tau_max=120.0,
    dt=0.01,
):
    """
    Vectorized RK4 calculation of T_out over a grid.

    Entries that do not cross x_out before tau_max remain np.nan.
    The exact scaling calculations below use solve_ivp rather than
    this grid solver.
    """
    x = np.array(x0, dtype=float, copy=True)
    nbar = np.array(nbar0, dtype=float, copy=True)

    pb = np.broadcast_to(
        np.asarray(phi_bar_values, dtype=float),
        x.shape,
    )

    T_out = np.full(x.shape, np.nan, dtype=float)
    active = x < x_out

    tau = 0.0
    n_steps = int(np.ceil(tau_max / dt))

    for _ in range(n_steps):
        if not np.any(active):
            break

        indices = np.flatnonzero(active)

        xa = x.flat[indices]
        na = nbar.flat[indices]
        pba = pb.flat[indices]

        k1x, k1n = latency_rhs_tau_array(
            xa, na, pba, ends, gamma, nu
        )

        k2x, k2n = latency_rhs_tau_array(
            xa + 0.5 * dt * k1x,
            na + 0.5 * dt * k1n,
            pba,
            ends,
            gamma,
            nu,
        )

        k3x, k3n = latency_rhs_tau_array(
            xa + 0.5 * dt * k2x,
            na + 0.5 * dt * k2n,
            pba,
            ends,
            gamma,
            nu,
        )

        k4x, k4n = latency_rhs_tau_array(
            xa + dt * k3x,
            na + dt * k3n,
            pba,
            ends,
            gamma,
            nu,
        )

        x_new = (
            xa
            + dt
            * (k1x + 2.0 * k2x + 2.0 * k3x + k4x)
            / 6.0
        )

        nbar_new = (
            na
            + dt
            * (k1n + 2.0 * k2n + 2.0 * k3n + k4n)
            / 6.0
        )

        crossed = (
            (xa < x_out)
            & (x_new >= x_out)
        )

        if np.any(crossed):
            crossing_fraction = (
                (x_out - xa[crossed])
                / (x_new[crossed] - xa[crossed])
            )

            T_out.flat[indices[crossed]] = (
                tau + dt * crossing_fraction
            )

        # Projection is used only for the coarse heatmap solver.
        x.flat[indices] = np.clip(x_new, 0.0, 1.0)
        nbar.flat[indices] = np.maximum(nbar_new, 0.0)

        active.flat[indices[crossed]] = False

        tau += dt

    return T_out


# ------------------------------------------------------------
# Awakening endpoint: exact interior roots and saddle-node
# ------------------------------------------------------------

def awakening_phi_sn(alpha, beta):
    """
    Biologically relevant saddle-node environmental load:

        phi_bar_SN =
        [3 beta - alpha - 2 sqrt(2 beta (beta-alpha))] / alpha

    This branch lies above phi_bar=1 when beta > 2 alpha.
    """
    if beta <= 2.0 * alpha:
        raise ValueError(
            "The above-threshold saddle-node requires beta > 2*alpha."
        )

    return (
        3.0 * beta
        - alpha
        - 2.0 * np.sqrt(
            2.0 * beta * (beta - alpha)
        )
    ) / alpha


def awakening_interior_roots(
    alpha,
    beta,
    phi_bar_values,
):
    """
    Interior equilibria for the coexistence-to-cancer endpoint.

    They solve

        phi_bar (beta-alpha) x^2
        + [alpha(phi_bar-1)-beta] x
        + alpha = 0.

    Returns x_minus, x_plus and the discriminant.
    Values outside (0,1) are returned as np.nan.
    """
    pb = np.asarray(phi_bar_values, dtype=float)

    A = pb * (beta - alpha)
    B = alpha * (pb - 1.0) - beta

    discriminant = (
        B**2
        - 4.0 * A * alpha
    )

    sqrt_discriminant = np.sqrt(
        np.maximum(discriminant, 0.0)
    )

    x_minus = (
        -B - sqrt_discriminant
    ) / (2.0 * A)

    x_plus = (
        -B + sqrt_discriminant
    ) / (2.0 * A)

    valid = discriminant >= 0.0

    x_minus = np.where(
        valid
        & (x_minus > 0.0)
        & (x_minus < 1.0),
        x_minus,
        np.nan,
    )

    x_plus = np.where(
        valid
        & (x_plus > 0.0)
        & (x_plus < 1.0),
        x_plus,
        np.nan,
    )

    return x_minus, x_plus, discriminant


# ------------------------------------------------------------
# Priming endpoint: stable manifold of the saddle
# ------------------------------------------------------------

def get_interior_saddle(case):
    """
    Locate the unique interior saddle using the existing
    compute_equilibria function.
    """
    equilibria = compute_equilibria(case)

    saddles = [
        equilibrium
        for equilibrium in equilibria
        if (
            equilibrium["kind"] == "saddle"
            and 0.0 < equilibrium["x"] < 1.0
        )
    ]

    if len(saddles) != 1:
        raise RuntimeError(
            "Expected exactly one interior saddle, "
            f"but found {len(saddles)}."
        )

    return np.array(
        [
            saddles[0]["x"],
            saddles[0]["nbar"],
        ],
        dtype=float,
    )


def stable_manifold_branches(
    case,
    nbar_max=2.5,
    epsilon=1e-7,
    integration_max=200.0,
):
    """
    Trace both branches of the saddle's stable manifold by
    integrating the reversed vector field from points displaced
    along the stable eigenvector.
    """
    saddle = get_interior_saddle(case)

    eigenvalues, eigenvectors = np.linalg.eig(
        jacobian_tau(
            case,
            saddle[0],
            saddle[1],
        )
    )

    stable_index = np.argmin(
        np.real(eigenvalues)
    )

    stable_vector = np.real(
        eigenvectors[:, stable_index]
    )

    stable_vector /= np.linalg.norm(
        stable_vector
    )

    pb = phi_bar(case)

    def reversed_rhs(s, y):
        return -latency_rhs_tau(
            s,
            y,
            case.ends,
            pb,
            case.gamma,
            case.nu,
        )

    def event_x_zero(s, y):
        return y[0]

    def event_x_one(s, y):
        return 1.0 - y[0]

    def event_nbar_zero(s, y):
        return y[1]

    def event_nbar_max(s, y):
        return nbar_max - y[1]

    for event in (
        event_x_zero,
        event_x_one,
        event_nbar_zero,
        event_nbar_max,
    ):
        event.terminal = True
        event.direction = -1

    branches = []

    for sign in (-1.0, 1.0):
        initial_state = (
            saddle
            + sign * epsilon * stable_vector
        )

        sol = solve_ivp(
            reversed_rhs,
            (0.0, integration_max),
            initial_state,
            events=(
                event_x_zero,
                event_x_one,
                event_nbar_zero,
                event_nbar_max,
            ),
            method="DOP853",
            rtol=1e-10,
            atol=1e-12,
            max_step=0.02,
        )

        branches.append(sol.y.T)

    return (
        saddle,
        eigenvalues,
        stable_vector,
        branches,
    )


# ------------------------------------------------------------
# Regression helper
# ------------------------------------------------------------

def linear_fit_with_r2(x, y):
    """
    Fit y = slope*x + intercept and return slope, intercept, R^2.
    """
    slope, intercept = np.polyfit(x, y, 1)

    prediction = (
        slope * x + intercept
    )

    residual_sum = np.sum(
        (y - prediction) ** 2
    )

    total_sum = np.sum(
        (y - np.mean(y)) ** 2
    )

    r_squared = (
        1.0 - residual_sum / total_sum
    )

    return slope, intercept, r_squared
    

# ============================================================
# Calculate bifurcation, latency, separatrix and timescale data
# ============================================================

latency_cases = build_default_cases()

awakening_case = latency_cases["awakening"]
priming_case = latency_cases["primed_threshold"]


# ------------------------------------------------------------
# Parameters used in the awakening endpoint
# ------------------------------------------------------------

alpha_aw = awakening_case.ends.alpha_h
beta_aw = awakening_case.ends.beta_h

phi_bar_sn = awakening_phi_sn(
    alpha_aw,
    beta_aw,
)

x_sn = (
    beta_aw
    - alpha_aw * (phi_bar_sn - 1.0)
) / (
    2.0
    * phi_bar_sn
    * (beta_aw - alpha_aw)
)

nbar_sn = phi_bar_sn * x_sn


# ------------------------------------------------------------
# Saddle-node scaling
# ------------------------------------------------------------

delta_phi = np.logspace(
    -4,
    -2,
    15,
)

T_saddle_node = np.array(
    [
        time_to_outgrowth(
            case_at_phi_bar(
                awakening_case,
                phi_bar_sn + delta,
            ),
            x0=0.10,
            nbar0=0.0,
            x_out=0.9,
            tau_max=5000.0,
            max_step=1.0,
        )
        for delta in delta_phi
    ]
)

(
    saddle_node_exponent,
    saddle_node_intercept,
    saddle_node_r2,
) = linear_fit_with_r2(
    np.log(delta_phi),
    np.log(T_saddle_node),
)


# ------------------------------------------------------------
# Priming saddle, separatrix and logarithmic scaling
# ------------------------------------------------------------

(
    priming_saddle,
    priming_eigenvalues,
    priming_stable_vector,
    separatrix_branches,
) = stable_manifold_branches(
    priming_case,
    nbar_max=2.5,
)

# Euclidean normal to the local stable-manifold tangent
separatrix_normal = np.array(
    [
        priming_stable_vector[1],
        -priming_stable_vector[0],
    ],
    dtype=float,
)

separatrix_normal /= np.linalg.norm(
    separatrix_normal
)

# Ensure the positive direction points into the cancer basin
test_time = time_to_outgrowth(
    priming_case,
    x0=(
        priming_saddle
        + 1e-3 * separatrix_normal
    )[0],
    nbar0=(
        priming_saddle
        + 1e-3 * separatrix_normal
    )[1],
    x_out=0.9,
    tau_max=500.0,
    max_step=0.1,
)

if not np.isfinite(test_time):
    separatrix_normal *= -1.0


lambda_u = float(
    np.max(
        np.real(
            priming_eigenvalues
        )
    )
)

separatrix_distances = np.logspace(
    -7,
    -2,
    20,
)

T_separatrix = np.array(
    [
        time_to_outgrowth(
            priming_case,
            x0=(
                priming_saddle
                + distance * separatrix_normal
            )[0],
            nbar0=(
                priming_saddle
                + distance * separatrix_normal
            )[1],
            x_out=0.9,
            tau_max=1000.0,
            rtol=1e-10,
            atol=1e-12,
            max_step=0.1,
        )
        for distance in separatrix_distances
    ]
)

(
    separatrix_slope,
    separatrix_intercept,
    separatrix_r2,
) = linear_fit_with_r2(
    -np.log(separatrix_distances),
    T_separatrix,
)


# ------------------------------------------------------------
# Awakening latency heatmap
# ------------------------------------------------------------

awakening_x0_values = np.linspace(
    0.01,
    0.89,
    90,
)

awakening_phi_bar_values = np.linspace(
    0.8,
    2.6,
    121,
)

(
    awakening_X0,
    awakening_PHI_BAR,
) = np.meshgrid(
    awakening_x0_values,
    awakening_phi_bar_values,
)

awakening_T_out = time_to_outgrowth_grid(
    x0=awakening_X0,
    nbar0=np.zeros_like(awakening_X0),
    phi_bar_values=awakening_PHI_BAR,
    ends=awakening_case.ends,
    gamma=awakening_case.gamma,
    nu=awakening_case.nu,
    x_out=0.9,
    tau_max=120.0,
    dt=0.01,
)


# ------------------------------------------------------------
# Priming latency heatmap
# ------------------------------------------------------------

priming_x0_values = np.linspace(
    0.01,
    0.89,
    101,
)

priming_nbar0_values = np.linspace(
    0.0,
    2.5,
    121,
)

(
    priming_X0,
    priming_NBAR0,
) = np.meshgrid(
    priming_x0_values,
    priming_nbar0_values,
)

priming_T_out = time_to_outgrowth_grid(
    x0=priming_X0,
    nbar0=priming_NBAR0,
    phi_bar_values=np.full_like(
        priming_X0,
        phi_bar(priming_case),
    ),
    ends=priming_case.ends,
    gamma=priming_case.gamma,
    nu=priming_case.nu,
    x_out=0.9,
    tau_max=80.0,
    dt=0.01,
)


# ------------------------------------------------------------
# Relative-timescale sensitivity
# ------------------------------------------------------------

epsilon_values = np.logspace(
    -2,
    1,
    30,
)

T_out_tau = np.empty_like(
    epsilon_values
)

for index, epsilon in enumerate(
    epsilon_values
):
    gamma_i = epsilon * alpha_aw

    # Hold phi_bar fixed while changing gamma:
    # phi = phi_bar * gamma * k
    timescale_case = Case(
        name="Timescale sensitivity",
        ends=awakening_case.ends,
        phi=(
            phi_bar(awakening_case)
            * gamma_i
            * awakening_case.k
        ),
        gamma=gamma_i,
        k=awakening_case.k,
        nu=awakening_case.nu,
        tmax=awakening_case.tmax,
        x0s=(0.10,),
        n0=0.0,
    )

    T_out_tau[index] = time_to_outgrowth(
        timescale_case,
        x0=0.10,
        nbar0=0.0,
        x_out=0.9,
        tau_max=5000.0,
        max_step=1.0,
    )

# alpha*t_out = T_out_tau / epsilon
normalized_dimensional_time = (
    T_out_tau / epsilon_values
)


# ------------------------------------------------------------
# Robustness to the operational outgrowth threshold
# ------------------------------------------------------------

outgrowth_thresholds = [
    0.80,
    0.90,
    0.95,
]

threshold_robustness = {}

for x_out_value in outgrowth_thresholds:
    threshold_times = np.array(
        [
            time_to_outgrowth(
                case_at_phi_bar(
                    awakening_case,
                    phi_bar_sn + delta,
                ),
                x0=0.10,
                nbar0=0.0,
                x_out=x_out_value,
                tau_max=5000.0,
                max_step=1.0,
            )
            for delta in delta_phi
        ]
    )

    (
        threshold_exponent,
        threshold_intercept,
        threshold_r2,
    ) = linear_fit_with_r2(
        np.log(delta_phi),
        np.log(threshold_times),
    )

    threshold_robustness[x_out_value] = {
        "times": threshold_times,
        "exponent": threshold_exponent,
        "intercept": threshold_intercept,
        "r2": threshold_r2,
    }


# ------------------------------------------------------------
# Validate grid solver against solve_ivp at representative points
# ------------------------------------------------------------

validation_points = [
    (0.10, 0.0, 2.01),
    (0.40, 0.0, 2.25),
    (0.80, 0.0, 2.50),
]

for x0_test, nbar0_test, pb_test in validation_points:
    exact_time = time_to_outgrowth(
        case_at_phi_bar(
            awakening_case,
            pb_test,
        ),
        x0=x0_test,
        nbar0=nbar0_test,
        x_out=0.9,
        tau_max=500.0,
    )

    grid_time = time_to_outgrowth_grid(
        x0=np.array([x0_test]),
        nbar0=np.array([nbar0_test]),
        phi_bar_values=np.array([pb_test]),
        ends=awakening_case.ends,
        gamma=awakening_case.gamma,
        nu=awakening_case.nu,
        x_out=0.9,
        tau_max=120.0,
        dt=0.01,
    )[0]

    difference = abs(
        exact_time - grid_time
    )

    print(
        f"Validation: x0={x0_test:.2f}, "
        f"phi_bar={pb_test:.2f}, "
        f"solve_ivp={exact_time:.6f}, "
        f"RK4={grid_time:.6f}, "
        f"difference={difference:.2e}"
    )

    assert difference < 1e-3


# ------------------------------------------------------------
# Print the central results
# ------------------------------------------------------------

print("\n--- Awakening saddle-node ---")
print(f"phi_bar_SN = {phi_bar_sn:.12f}")
print(f"x_SN       = {x_sn:.12f}")
print(f"nbar_SN    = {nbar_sn:.12f}")

print("\n--- Saddle-node latency fit ---")
print(
    "exponent = "
    f"{saddle_node_exponent:.6f}"
)
print(
    "R^2      = "
    f"{saddle_node_r2:.8f}"
)

print("\n--- Priming saddle ---")
print(
    "saddle = "
    f"({priming_saddle[0]:.12f}, "
    f"{priming_saddle[1]:.12f})"
)
print(
    "eigenvalues = ",
    priming_eigenvalues,
)
print(f"lambda_u      = {lambda_u:.8f}")
print(f"1/lambda_u    = {1.0/lambda_u:.8f}")

print("\n--- Separatrix latency fit ---")
print(
    "fitted slope = "
    f"{separatrix_slope:.8f}"
)
print(
    "R^2          = "
    f"{separatrix_r2:.10f}"
)

print("\n--- Threshold robustness ---")
for threshold, values in threshold_robustness.items():
    print(
        f"x_out={threshold:.2f}: "
        f"exponent={values['exponent']:.6f}, "
        f"R^2={values['r2']:.8f}"
    )

print("\n--- Heatmap ranges ---")
print(
    "Awakening finite T_out range:",
    np.nanmin(awakening_T_out),
    np.nanmax(awakening_T_out),
)
print(
    "Priming finite T_out range:",
    np.nanmin(priming_T_out),
    np.nanmax(priming_T_out),
)

# ============================================================
# Main figure:
# A. Awakening bifurcation diagram
# B. Awakening time-to-outgrowth map
# C. Priming time-to-outgrowth map
# ============================================================
import copy

phi_bar_branch = np.linspace(
    0.05,
    2.60,
    1000,
)

(
    x_minus_branch,
    x_plus_branch,
    branch_discriminant,
) = awakening_interior_roots(
    alpha_aw,
    beta_aw,
    phi_bar_branch,
)

"""
latency_cmap = plt.get_cmap(
    "viridis"
).copy()

# Grey denotes no observed outgrowth before the integration limit
latency_cmap.set_bad("#D9D9D9")
"""

latency_cmap = copy.copy(plt.get_cmap("viridis"))

# Grey denotes no observed outgrowth before the integration limit
latency_cmap.set_bad("#D9D9D9")

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16.0, 4.8),
    constrained_layout=True,
)


# ------------------------------------------------------------
# Panel A: bifurcation diagram
# ------------------------------------------------------------

ax = axes[0]

below_load_threshold = (
    phi_bar_branch < 1.0
)

above_load_threshold = (
    phi_bar_branch > 1.0
)

# x=0 is a saddle throughout the awakening endpoint
ax.plot(
    phi_bar_branch,
    np.zeros_like(phi_bar_branch),
    "--",
    color="black",
    linewidth=1.6,
    label="boundary saddle",
)

# x=1 is a saddle below phi_bar=1
ax.plot(
    phi_bar_branch[below_load_threshold],
    np.ones(
        np.sum(below_load_threshold)
    ),
    "--",
    color=COL["cancer"],
    linewidth=2.0,
)

# x=1 is stable above phi_bar=1
ax.plot(
    phi_bar_branch[above_load_threshold],
    np.ones(
        np.sum(above_load_threshold)
    ),
    "-",
    color=COL["cancer"],
    linewidth=2.5,
    label="cancer-dominant stable",
)

# Stable dormant-like interior branch
ax.plot(
    phi_bar_branch,
    x_minus_branch,
    "-",
    color=COL["coexist"],
    linewidth=2.5,
    label="stable interior",
)

# Interior saddle branch
ax.plot(
    phi_bar_branch,
    x_plus_branch,
    "--",
    color=COL["bistab"],
    linewidth=2.2,
    label="interior saddle",
)

# Environmental-load threshold
ax.axvline(
    1.0,
    color="0.4",
    linestyle=":",
    linewidth=1.5,
)

# Saddle-node threshold
ax.axvline(
    phi_bar_sn,
    color="black",
    linestyle=":",
    linewidth=1.8,
)

# Representative awakening parameter from the original awakening example
ax.axvline(
    phi_bar(awakening_case),
    color=COL["cancer"],
    linestyle="-.",
    linewidth=1.8,
)

ax.plot(
    phi_bar_sn,
    x_sn,
    marker="o",
    markersize=6,
    color="black",
)

ax.text(
    1.0,
    0.03,
    r"$\bar\phi=1$",
    rotation=90,
    ha="right",
    va="bottom",
    fontsize=9,
)

ax.text(
    phi_bar_sn,
    0.03,
    r"$\bar\phi_{\rm SN}$",
    rotation=90,
    ha="right",
    va="bottom",
    fontsize=9,
)

ax.text(
    phi_bar(awakening_case),
    0.65,
    "representative\nawakening",
    rotation=90,
    ha="right",
    va="bottom",
    fontsize=9,
    color=COL["cancer"],
)

ax.set_xlim(
    0.0,
    2.6,
)

ax.set_ylim(
    -0.03,
    1.03,
)

ax.set_xlabel(
    r"Environmental load $\bar\phi$",
    fontsize=13,
)

ax.set_ylabel(
    r"Equilibrium cancer fraction $x^*$",
    fontsize=13,
)

ax.set_title(
    "Awakening bifurcation",
    fontsize=15,
)

ax.tick_params(
    labelsize=10,
)

ax.legend(
    fontsize=8,
    loc="center left",
)


# ------------------------------------------------------------
# Panel B: awakening latency heatmap
# ------------------------------------------------------------

ax = axes[1]

awakening_masked = np.ma.masked_invalid(
    awakening_T_out
)

awakening_map = ax.pcolormesh(
    awakening_phi_bar_values,
    awakening_x0_values,
    awakening_masked.T,
    shading="auto",
    cmap=latency_cmap,
    norm=LogNorm(
        vmin=max(
            np.nanmin(awakening_T_out),
            1e-2,
        ),
        vmax=np.nanmax(
            awakening_T_out
        ),
    ),
)

ax.axvline(
    phi_bar_sn,
    color="white",
    linestyle="--",
    linewidth=1.7,
)

ax.axvline(
    phi_bar(awakening_case),
    color="white",
    linestyle=":",
    linewidth=1.7,
)

ax.set_xlabel(
    r"Environmental load $\bar\phi$",
    fontsize=13,
)

ax.set_ylabel(
    r"Initial cancer fraction $x_0$",
    fontsize=13,
)

ax.set_title(
    r"Awakening $T_{\rm out}$",
    fontsize=15,
)

ax.tick_params(
    labelsize=10,
)

awakening_colorbar = fig.colorbar(
    awakening_map,
    ax=ax,
)

awakening_colorbar.set_label(
    r"$T_{\rm out}$ in $\tau$",
    fontsize=11,
)


# Saddle-node scaling inset
inset = ax.inset_axes(
    [0.50, 0.12, 0.45, 0.38]
)

inset.loglog(
    delta_phi,
    T_saddle_node,
    "o",
    color="black",
    markersize=3,
)

inset.loglog(
    delta_phi,
    np.exp(
        saddle_node_intercept
    )
    * delta_phi**saddle_node_exponent,
    "-",
    color=COL["cancer"],
    linewidth=1.2,
)

inset.text(
    0.05,
    0.08,
    rf"slope $={saddle_node_exponent:.3f}$",
    transform=inset.transAxes,
    fontsize=7,
)

inset.set_xlabel(
    r"$\bar\phi-\bar\phi_{\rm SN}$",
    fontsize=7,
)

inset.set_ylabel(
    r"$T_{\rm out}$",
    fontsize=7,
)

inset.tick_params(
    labelsize=6,
)


# ------------------------------------------------------------
# Panel C: priming latency heatmap
# ------------------------------------------------------------

ax = axes[2]

priming_masked = np.ma.masked_invalid(
    priming_T_out
)

priming_map = ax.pcolormesh(
    priming_x0_values,
    priming_nbar0_values,
    priming_masked,
    shading="auto",
    cmap=latency_cmap,
    norm=LogNorm(
        vmin=max(
            np.nanmin(priming_T_out),
            0.1,
        ),
        vmax=np.nanmax(
            priming_T_out
        ),
    ),
)

# Exact stable manifold from backward integration
for branch in separatrix_branches:
    ax.plot(
        branch[:, 0],
        branch[:, 1],
        color="white",
        linewidth=2.2,
    )

ax.plot(
    priming_saddle[0],
    priming_saddle[1],
    marker="*",
    markersize=11,
    markerfacecolor="white",
    markeredgecolor="black",
    linestyle="None",
)

ax.set_xlabel(
    r"Initial cancer fraction $x_0$",
    fontsize=13,
)

ax.set_ylabel(
    r"Initial conditioning $\bar n_0$",
    fontsize=13,
)

ax.set_title(
    r"Priming $T_{\rm out}$",
    fontsize=15,
)

ax.tick_params(
    labelsize=10,
)

priming_colorbar = fig.colorbar(
    priming_map,
    ax=ax,
)

priming_colorbar.set_label(
    r"$T_{\rm out}$ in $\tau$",
    fontsize=11,
)


# Separatrix scaling inset
inset = ax.inset_axes(
    [0.52, 0.12, 0.43, 0.38]
)

inset.semilogx(
    separatrix_distances,
    T_separatrix,
    "o",
    color="black",
    markersize=3,
)

inset.semilogx(
    separatrix_distances,
    (
        separatrix_slope
        * (-np.log(separatrix_distances))
        + separatrix_intercept
    ),
    "-",
    color=COL["bistab"],
    linewidth=1.2,
)

inset.text(
    0.05,
    0.08,
    (
        rf"slope $={separatrix_slope:.3f}$"
        "\n"
        rf"$1/\lambda_u={1.0/lambda_u:.3f}$"
    ),
    transform=inset.transAxes,
    fontsize=7,
)

inset.set_xlabel(
    r"Distance $d$",
    fontsize=7,
)

inset.set_ylabel(
    r"$T_{\rm out}$",
    fontsize=7,
)

inset.tick_params(
    labelsize=6,
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

main_figure_svg = os.path.join(
    LATENCY_OUTDIR,
    "latency_analysis_main.svg",
)

main_figure_png = os.path.join(
    LATENCY_OUTDIR,
    "latency_analysis_main.png",
)

fig.savefig(
    main_figure_svg,
    bbox_inches="tight",
)

fig.savefig(
    main_figure_png,
    dpi=450,
    bbox_inches="tight",
)

plt.show()

print(main_figure_svg)
print(main_figure_png)

In [ ]:
# ============================================================
# Supplementary figure:
# A. Relative-timescale sensitivity
# B. Robustness to x_out
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(9.5, 4.2),
    constrained_layout=True,
)


# ------------------------------------------------------------
# Panel A: relative-timescale sensitivity
# ------------------------------------------------------------

ax = axes[0]

ax.loglog(
    epsilon_values,
    T_out_tau,
    marker="o",
    linewidth=2.0,
    label=r"$T_{\rm out}^{(\tau)}$",
)

ax.loglog(
    epsilon_values,
    normalized_dimensional_time,
    marker="s",
    linewidth=2.0,
    label=r"$\alpha t_{\rm out}$",
)

current_epsilon = (
    awakening_case.gamma
    / alpha_aw
)

ax.axvline(
    current_epsilon,
    color="black",
    linestyle=":",
    linewidth=1.8,
    label="representative awakening example",
)

ax.set_xlabel(
    r"$\varepsilon=\gamma/\alpha$",
    fontsize=13,
)

ax.set_ylabel(
    "Latency",
    fontsize=13,
)

ax.set_title(
    "Relative-timescale sensitivity",
    fontsize=15,
)

ax.tick_params(
    labelsize=10,
)

ax.legend(
    fontsize=9,
)


# ------------------------------------------------------------
# Panel B: operational-threshold robustness
# ------------------------------------------------------------

ax = axes[1]

markers = {
    0.80: "o",
    0.90: "s",
    0.95: "^",
}

for threshold in outgrowth_thresholds:
    values = threshold_robustness[
        threshold
    ]

    ax.loglog(
        delta_phi,
        values["times"],
        marker=markers[threshold],
        markersize=4,
        linewidth=1.8,
        label=(
            rf"$x_{{\rm out}}={threshold:.2f}$, "
            rf"$m={values['exponent']:.3f}$"
        ),
    )

ax.set_xlabel(
    r"$\bar\phi-\bar\phi_{\rm SN}$",
    fontsize=13,
)

ax.set_ylabel(
    r"$T_{\rm out}$",
    fontsize=13,
)

ax.set_title(
    "Outgrowth-threshold robustness",
    fontsize=15,
)

ax.tick_params(
    labelsize=10,
)

ax.legend(
    fontsize=8,
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

supp_figure_svg = os.path.join(
    LATENCY_OUTDIR,
    "latency_analysis_supp.svg",
)

supp_figure_png = os.path.join(
    LATENCY_OUTDIR,
    "latency_analysis_supp.png",
)

fig.savefig(
    supp_figure_svg,
    bbox_inches="tight",
)

fig.savefig(
    supp_figure_png,
    dpi=450,
    bbox_inches="tight",
)

plt.show()

print(supp_figure_svg)
print(supp_figure_png)


# Save numerical arrays for reproducibility
data_file = os.path.join(
    LATENCY_OUTDIR,
    "latency_analysis_data.npz",
)

np.savez(
    data_file,
    phi_bar_sn=phi_bar_sn,
    x_sn=x_sn,
    nbar_sn=nbar_sn,
    delta_phi=delta_phi,
    T_saddle_node=T_saddle_node,
    saddle_node_exponent=saddle_node_exponent,
    saddle_node_r2=saddle_node_r2,
    priming_saddle=priming_saddle,
    priming_eigenvalues=priming_eigenvalues,
    separatrix_distances=separatrix_distances,
    T_separatrix=T_separatrix,
    separatrix_slope=separatrix_slope,
    separatrix_r2=separatrix_r2,
    awakening_x0_values=awakening_x0_values,
    awakening_phi_bar_values=awakening_phi_bar_values,
    awakening_T_out=awakening_T_out,
    priming_x0_values=priming_x0_values,
    priming_nbar0_values=priming_nbar0_values,
    priming_T_out=priming_T_out,
    epsilon_values=epsilon_values,
    T_out_tau=T_out_tau,
    normalized_dimensional_time=normalized_dimensional_time,
)

print(data_file)

### R1M3

In [ ]:
"""
R1M3 implementation: impulsive multi-wave seeding extension for the cancer--microenvironment model.

Notebook placement
------------------
Paste/run this cell after the final nondimensional core-model cell in
Paper_corrections.ipynb, i.e. after Case, build_default_cases, rhs_tau,
compute_equilibria, jacobian_tau, phi_bar, and alpha_beta_bar are defined.

Implemented audit fixes
-----------------------
- local `import copy` for robust partial reruns.
- the q--Delta tau grid contains the representative example exactly.
- an explicit RK4/adaptive-trajectory consistency check verifies that the
  heatmap gives N_seed=7 at q=0.08, Delta tau=1.0.

Outputs
-------
Figures/multiwave_priming.png and .svg
Figures/multiwave_initial_condition_sensitivity.png and .svg
Figures/multiwave_priming_data.npz
Figures/multiwave_representative_pulses.csv
"""

import os
import csv
import copy
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.integrate import solve_ivp

# -----------------------------------------------------------------------------
# 0. Parent-notebook checks
# -----------------------------------------------------------------------------
_REQUIRED = [
    "Case", "build_default_cases", "phi_bar", "rhs_tau",
    "jacobian_tau", "compute_equilibria", "alpha_beta_bar"
]
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run the final nondimensional core-model cell in Paper.ipynb first. "
        f"Missing: {_missing}"
    )

if "COL" not in globals():
    COL = {
        "host": "#707071",
        "coexist": "#629FB7",
        "bistab": "#106E91",
        "cancer": "#AF4772",
    }

# -----------------------------------------------------------------------------
# 1. Analysis configuration
# -----------------------------------------------------------------------------
OUTDIR = "Figures"
os.makedirs(OUTDIR, exist_ok=True)

CASES = build_default_cases()
PRIMING_CASE = CASES["primed_threshold"]

# Representative example used in panels A--B.
EXAMPLE_Q = 0.08
EXAMPLE_DELTA_TAU = 1.0
EXAMPLE_X0 = 0.0
EXAMPLE_NBAR0 = 0.0

# Operational definitions.
X_OUT = 0.90
MAX_WAVES = 30
POST_TAU_MAX = 200.0

# Publication grids.
Q_VALUES = np.unique(np.r_[np.linspace(0.02, 0.36, 61), EXAMPLE_Q])
DELTA_TAU_VALUES = np.unique(np.r_[np.linspace(0.05, 5.0, 61), EXAMPLE_DELTA_TAU])

# Initial-condition sensitivity at a representative inter-wave interval.
SENSITIVITY_DELTA_TAU = 1.0
SENSITIVITY_Q_VALUES = np.linspace(0.01, 0.36, 71)
SENSITIVITY_NBAR0_VALUES = np.linspace(0.0, 1.5, 61)

# Fixed-step RK4 resolution for fast batch calculations.
RK4_MAX_STEP = 0.05

# -----------------------------------------------------------------------------
# 2. Stable manifold of the saddle: the separatrix
# -----------------------------------------------------------------------------
def compute_separatrix(case: Case,
                       eps: float = 1e-7,
                       tau_back: float = 50.0,
                       max_step: float = 0.01):
    """Trace both stable-manifold branches backward from the saddle."""
    pb = phi_bar(case)
    saddles = [e for e in compute_equilibria(case) if e["kind"] == "saddle"]
    if len(saddles) != 1:
        raise RuntimeError(f"Expected one saddle, found {len(saddles)}.")

    saddle = saddles[0]
    y_star = np.array([saddle["x"], saddle["nbar"]], dtype=float)
    J = jacobian_tau(case, y_star[0], y_star[1])
    eigvals, eigvecs = np.linalg.eig(J)
    stable_idx = np.where(np.real(eigvals) < 0.0)[0]
    if len(stable_idx) != 1:
        raise RuntimeError("Saddle must have one stable eigenvalue.")

    idx = stable_idx[0]
    stable_eigenvalue = float(np.real(eigvals[idx]))
    stable_vector = np.real(eigvecs[:, idx])
    stable_vector /= np.linalg.norm(stable_vector)

    branches = []
    for sign in (-1.0, +1.0):
        y0 = y_star + sign * eps * stable_vector

        def event_x0(_tau, y, *_args): return y[0]
        def event_x1(_tau, y, *_args): return 1.0 - y[0]
        def event_n0(_tau, y, *_args): return y[1]
        def event_nmax(_tau, y, *_args): return pb - y[1]

        events = [event_x0, event_x1, event_n0, event_nmax]
        for event in events:
            event.terminal = True
            event.direction = 0

        sol = solve_ivp(
            rhs_tau,
            (0.0, -tau_back),
            y0,
            args=(case.ends, pb, case.gamma, case.nu),
            events=events,
            rtol=1e-10,
            atol=1e-12,
            max_step=max_step,
        )
        if not sol.success:
            raise RuntimeError(sol.message)
        branches.append(sol.y.copy())

    x_all = np.concatenate([b[0] for b in branches])
    n_all = np.concatenate([b[1] for b in branches])
    order = np.argsort(n_all)
    x_all = x_all[order]
    n_all = n_all[order]

    # Remove near-duplicate n values around the saddle.
    _, unique_idx = np.unique(np.round(n_all, 12), return_index=True)
    unique_idx = np.sort(unique_idx)
    x_all = x_all[unique_idx]
    n_all = n_all[unique_idx]

    return {
        "x": x_all,
        "nbar": n_all,
        "saddle": y_star,
        "stable_eigenvalue": stable_eigenvalue,
        "stable_eigenvector": stable_vector,
        "branches": branches,
    }


def evaluate_separatrix(separatrix, nbar):
    """Evaluate x_sep(nbar), clamping to the computed physical interval."""
    nbar = np.asarray(nbar, dtype=float)
    n_clamped = np.clip(nbar, separatrix["nbar"][0], separatrix["nbar"][-1])
    return np.interp(n_clamped, separatrix["nbar"], separatrix["x"])


SEPARATRIX = compute_separatrix(PRIMING_CASE)

# -----------------------------------------------------------------------------
# 3. Pulse map and exact threshold formulae
# -----------------------------------------------------------------------------
def pulse_jump_q(x_minus, q):
    """x^+ = x^- + q(1-x^-), with 0 <= q < 1."""
    x_minus = np.asarray(x_minus, dtype=float)
    q = np.asarray(q, dtype=float)
    if np.any((q < 0.0) | (q >= 1.0)):
        raise ValueError("q must satisfy 0 <= q < 1.")
    return x_minus + q * (1.0 - x_minus)


def pulse_jump_eta(x_minus, eta):
    """Equivalent count-based map x^+=(x^-+eta)/(1+eta)."""
    x_minus = np.asarray(x_minus, dtype=float)
    eta = np.asarray(eta, dtype=float)
    if np.any(eta < 0.0):
        raise ValueError("eta must be non-negative.")
    return (x_minus + eta) / (1.0 + eta)


def q_from_eta(eta):
    eta = np.asarray(eta, dtype=float)
    return eta / (1.0 + eta)


def eta_from_q(q):
    q = np.asarray(q, dtype=float)
    return q / (1.0 - q)


def critical_wave_size_q(separatrix, x_minus, nbar_minus):
    """Exact q needed to land on the separatrix from (x^-, nbar^-)."""
    x_minus = np.asarray(x_minus, dtype=float)
    x_sep = evaluate_separatrix(separatrix, nbar_minus)
    with np.errstate(divide="ignore", invalid="ignore"):
        q_c = (x_sep - x_minus) / (1.0 - x_minus)
    q_c = np.where(x_minus >= x_sep, 0.0, q_c)
    return np.clip(q_c, 0.0, 1.0)


def critical_wave_size_eta(separatrix, x_minus, nbar_minus):
    """Equivalent eta threshold: eta_c=(x_sep-x^-)/(1-x_sep)."""
    x_minus = np.asarray(x_minus, dtype=float)
    x_sep = evaluate_separatrix(separatrix, nbar_minus)
    with np.errstate(divide="ignore", invalid="ignore"):
        eta_c = (x_sep - x_minus) / (1.0 - x_sep)
    eta_c = np.where(x_minus >= x_sep, 0.0, eta_c)
    return np.maximum(eta_c, 0.0)


def rapid_seeding_wave_count(separatrix, q, x0=0.0, nbar0=0.0):
    """Exact N_seed as Delta tau -> 0: x_m=1-(1-x0)(1-q)^m."""
    x_sep = float(evaluate_separatrix(separatrix, nbar0))
    if x0 >= x_sep:
        return 0
    if q <= 0.0:
        return np.inf
    ratio = (1.0 - x_sep) / (1.0 - x0)
    return int(math.floor(math.log(ratio) / math.log(1.0 - q)) + 1)

# -----------------------------------------------------------------------------
# 4. Accurate single-trajectory solver for the representative example
# -----------------------------------------------------------------------------
def integrate_interval_solve_ivp(case, y0, delta_tau, record=False, npts=120):
    y0 = np.asarray(y0, dtype=float)
    if delta_tau == 0.0:
        if record:
            return np.array([0.0]), y0.reshape(2, 1)
        return y0.copy()
    t_eval = np.linspace(0.0, delta_tau, npts) if record else None
    sol = solve_ivp(
        rhs_tau,
        (0.0, delta_tau),
        y0,
        t_eval=t_eval,
        args=(case.ends, phi_bar(case), case.gamma, case.nu),
        rtol=1e-9,
        atol=1e-11,
        max_step=min(0.03, delta_tau / 10.0),
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return (sol.t, sol.y) if record else sol.y[:, -1]


def autonomous_time_to_outgrowth_solve_ivp(case, y0, x_out=X_OUT,
                                            tau_max=POST_TAU_MAX,
                                            record=False):
    y0 = np.asarray(y0, dtype=float)
    if y0[0] >= x_out:
        if record:
            return 0.0, np.array([0.0]), y0.reshape(2, 1)
        return 0.0

    def event(_tau, y, *_args): return y[0] - x_out
    event.terminal = True
    event.direction = +1

    sol = solve_ivp(
        rhs_tau,
        (0.0, tau_max),
        y0,
        args=(case.ends, phi_bar(case), case.gamma, case.nu),
        events=event,
        rtol=1e-9,
        atol=1e-11,
        max_step=0.03,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    if len(sol.t_events[0]) == 0:
        if record:
            return np.nan, sol.t, sol.y
        return np.nan

    tau_out = float(sol.t_events[0][0])
    if not record:
        return tau_out

    t_eval = np.linspace(0.0, tau_out, 600)
    plot_sol = solve_ivp(
        rhs_tau,
        (0.0, tau_out),
        y0,
        t_eval=t_eval,
        args=(case.ends, phi_bar(case), case.gamma, case.nu),
        rtol=1e-9,
        atol=1e-11,
        max_step=0.03,
    )
    return tau_out, plot_sol.t, plot_sol.y


def simulate_pulse_train(case, separatrix, q, delta_tau,
                         x0=0.0, nbar0=0.0,
                         max_waves=MAX_WAVES,
                         x_out=X_OUT,
                         post_tau_max=POST_TAU_MAX,
                         record=False):
    """Accurate hybrid simulation of one deterministic pulse train."""
    state = np.array([x0, nbar0], dtype=float)
    tau_now = 0.0
    pulse_times, pulse_pre, pulse_post, segments = [], [], [], []
    n_seed = None
    tau_post = np.nan
    tau_out = np.nan

    for wave in range(1, max_waves + 1):
        before = state.copy()
        after = np.array([pulse_jump_q(before[0], q), before[1]], dtype=float)
        pulse_times.append(tau_now)
        pulse_pre.append(before)
        pulse_post.append(after)
        state = after

        if state[0] > float(evaluate_separatrix(separatrix, state[1])):
            n_seed = wave
            if record:
                tau_post, local_t, local_y = autonomous_time_to_outgrowth_solve_ivp(
                    case, state, x_out=x_out, tau_max=post_tau_max, record=True
                )
                segments.append({
                    "kind": "post_success",
                    "tau": tau_now + local_t,
                    "x": local_y[0],
                    "nbar": local_y[1],
                })
            else:
                tau_post = autonomous_time_to_outgrowth_solve_ivp(
                    case, state, x_out=x_out, tau_max=post_tau_max, record=False
                )
            if np.isfinite(tau_post):
                tau_out = tau_now + tau_post
            break

        if wave < max_waves:
            if record:
                local_t, local_y = integrate_interval_solve_ivp(
                    case, state, delta_tau, record=True,
                    npts=max(80, int(120 * delta_tau) + 2)
                )
                segments.append({
                    "kind": "inter_wave",
                    "tau": tau_now + local_t,
                    "x": local_y[0],
                    "nbar": local_y[1],
                })
                state = local_y[:, -1]
            else:
                state = integrate_interval_solve_ivp(case, state, delta_tau)
            tau_now += delta_tau

    return {
        "success": n_seed is not None,
        "n_seed": n_seed,
        "tau_out": tau_out,
        "t_out": tau_out / case.gamma if np.isfinite(tau_out) else np.nan,
        "tau_post": tau_post,
        "pulse_times": np.asarray(pulse_times),
        "pulse_pre": np.asarray(pulse_pre),
        "pulse_post": np.asarray(pulse_post),
        "segments": segments,
        "final_state": state.copy(),
        "q": q,
        "delta_tau": delta_tau,
        "x0": x0,
        "nbar0": nbar0,
    }

# -----------------------------------------------------------------------------
# 5. Fast vectorized RK4 for large parameter grids
# -----------------------------------------------------------------------------
def rhs_tau_array(case, x, nbar):
    a, b, _ = alpha_beta_bar(nbar, case.ends, case.nu)
    dx = (1.0 / case.gamma) * x * (1.0 - x) * (a - (a + b) * x)
    dn = phi_bar(case) * x - nbar
    return dx, dn


def rk4_advance_variable_durations(case, x, nbar, durations,
                                   max_step=RK4_MAX_STEP):
    """
    Advance many states by different durations in one vectorized RK4 loop.

    durations is broadcast to the state shape. States with duration zero are
    left unchanged. This avoids a slow Python loop over each Delta tau value.
    """
    x, nbar, durations = np.broadcast_arrays(
        np.asarray(x, dtype=float),
        np.asarray(nbar, dtype=float),
        np.asarray(durations, dtype=float),
    )
    x = x.copy()
    nbar = nbar.copy()
    remaining = np.maximum(durations.copy(), 0.0)

    while np.any(remaining > 1e-14):
        h = np.minimum(max_step, remaining)

        k1x, k1n = rhs_tau_array(case, x, nbar)
        k2x, k2n = rhs_tau_array(case, x + 0.5*h*k1x, nbar + 0.5*h*k1n)
        k3x, k3n = rhs_tau_array(case, x + 0.5*h*k2x, nbar + 0.5*h*k2n)
        k4x, k4n = rhs_tau_array(case, x + h*k3x, nbar + h*k3n)

        x += (h/6.0) * (k1x + 2*k2x + 2*k3x + k4x)
        nbar += (h/6.0) * (k1n + 2*k2n + 2*k3n + k4n)
        x = np.clip(x, 0.0, 1.0)
        nbar = np.maximum(nbar, 0.0)
        remaining = np.maximum(remaining - h, 0.0)

    return x, nbar


def pulse_batch_until_crossing(case, separatrix, q, delta_tau,
                               x0, nbar0, max_waves=MAX_WAVES):
    """
    Vectorized finite pulse train.

    q, delta_tau, x0 and nbar0 may all be arrays and are broadcast to a common
    shape. This allows the full (q, Delta tau) map to be computed at once.
    """
    q, delta_tau, x, nbar = np.broadcast_arrays(
        np.asarray(q, dtype=float),
        np.asarray(delta_tau, dtype=float),
        np.asarray(x0, dtype=float),
        np.asarray(nbar0, dtype=float),
    )
    q = q.copy()
    delta_tau = delta_tau.copy()
    x = x.copy()
    nbar = nbar.copy()
    shape = x.shape

    n_seed = np.full(shape, np.nan)
    crossing_x = np.full(shape, np.nan)
    crossing_nbar = np.full(shape, np.nan)
    crossing_tau = np.full(shape, np.nan)
    active = np.ones(shape, dtype=bool)

    for wave in range(1, max_waves + 1):
        x[active] = pulse_jump_q(x[active], q[active])
        x_sep = evaluate_separatrix(separatrix, nbar)
        crossed = active & (x > x_sep)

        n_seed[crossed] = wave
        crossing_x[crossed] = x[crossed]
        crossing_nbar[crossed] = nbar[crossed]
        crossing_tau[crossed] = (wave - 1) * delta_tau[crossed]
        active[crossed] = False

        if not np.any(active) or wave == max_waves:
            break

        durations = np.where(active, delta_tau, 0.0)
        x, nbar = rk4_advance_variable_durations(
            case, x, nbar, durations, max_step=RK4_MAX_STEP
        )

    return {
        "n_seed": n_seed,
        "crossing_x": crossing_x,
        "crossing_nbar": crossing_nbar,
        "crossing_tau": crossing_tau,
        "success": np.isfinite(n_seed),
    }

def batch_time_to_outgrowth(case, x0, nbar0, x_out=X_OUT,
                             tau_max=POST_TAU_MAX,
                             max_step=RK4_MAX_STEP):
    """Vectorized autonomous first-passage time after basin crossing."""
    x = np.asarray(x0, dtype=float).copy()
    nbar = np.asarray(nbar0, dtype=float).copy()
    shape = x.shape
    tau = np.full(shape, np.nan)
    valid = np.isfinite(x) & np.isfinite(nbar)
    immediate = valid & (x >= x_out)
    tau[immediate] = 0.0
    active = valid & ~immediate

    n_steps = int(math.ceil(tau_max / max_step))
    h = tau_max / n_steps
    elapsed = 0.0

    for _ in range(n_steps):
        if not np.any(active):
            break
        x_old = x[active].copy()
        xa, na = rk4_advance_variable_durations(case, x[active], nbar[active], h, max_step=h)
        elapsed += h

        crossed_local = (x_old < x_out) & (xa >= x_out)
        active_indices = np.flatnonzero(active)
        if np.any(crossed_local):
            idx = active_indices[crossed_local]
            # Linear interpolation within the final RK4 step.
            frac = (x_out - x_old[crossed_local]) / (xa[crossed_local] - x_old[crossed_local])
            tau.flat[idx] = elapsed - h + h * frac

        x[active] = xa
        nbar[active] = na
        active.flat[active_indices[crossed_local]] = False

    return tau


def compute_multiwave_grid_fast(case, separatrix, q_values, delta_values,
                                x0=0.0, nbar0=0.0,
                                max_waves=MAX_WAVES,
                                x_out=X_OUT):
    q_values = np.asarray(q_values, dtype=float)
    delta_values = np.asarray(delta_values, dtype=float)
    Q, DELTA = np.meshgrid(q_values, delta_values)

    result = pulse_batch_until_crossing(
        case, separatrix,
        q=Q,
        delta_tau=DELTA,
        x0=np.full_like(Q, x0),
        nbar0=np.full_like(Q, nbar0),
        max_waves=max_waves,
    )

    tau_post = batch_time_to_outgrowth(
        case, result["crossing_x"], result["crossing_nbar"],
        x_out=x_out, tau_max=POST_TAU_MAX, max_step=RK4_MAX_STEP
    )
    tau_out = result["crossing_tau"] + tau_post

    return {
        "q": q_values,
        "delta_tau": delta_values,
        "n_seed": result["n_seed"],
        "crossing_x": result["crossing_x"],
        "crossing_nbar": result["crossing_nbar"],
        "crossing_tau": result["crossing_tau"],
        "tau_post": tau_post,
        "tau_out": tau_out,
        "t_out": tau_out / case.gamma,
        "x0": x0,
        "nbar0": nbar0,
        "max_waves": max_waves,
        "x_out": x_out,
    }

def compute_initial_condition_grid_fast(case, separatrix, q_values,
                                        nbar0_values,
                                        delta_tau=SENSITIVITY_DELTA_TAU,
                                        x0=0.0,
                                        max_waves=MAX_WAVES):
    Q, N0 = np.meshgrid(q_values, nbar0_values)
    result = pulse_batch_until_crossing(
        case, separatrix,
        q=Q,
        delta_tau=delta_tau,
        x0=np.full_like(Q, x0),
        nbar0=N0,
        max_waves=max_waves,
    )
    return {
        "q": np.asarray(q_values),
        "nbar0": np.asarray(nbar0_values),
        "delta_tau": delta_tau,
        "n_seed": result["n_seed"],
        "max_waves": max_waves,
    }


# -----------------------------------------------------------------------------
# 5b. Consistency checks
# -----------------------------------------------------------------------------
def check_representative_grid_consistency(grid, example_result,
                                          q=EXAMPLE_Q,
                                          delta_tau=EXAMPLE_DELTA_TAU):
    """
    Ensure that the parameter-grid heatmap contains the representative example
    and gives the same integer wave count as the adaptive solve_ivp trajectory.

    This guards against RK4/adaptive-solver disagreement near the separatrix and
    against plotting a heatmap that does not contain the example point exactly.
    """
    q_idx = np.flatnonzero(np.isclose(grid["q"], q, rtol=0.0, atol=1e-12))
    d_idx = np.flatnonzero(np.isclose(grid["delta_tau"], delta_tau,
                                      rtol=0.0, atol=1e-12))
    if len(q_idx) != 1 or len(d_idx) != 1:
        raise RuntimeError(
            "The multiwave grid does not contain the representative point "
            f"q={q}, Delta tau={delta_tau} exactly."
        )
    n_grid = grid["n_seed"][d_idx[0], q_idx[0]]
    n_example = example_result["n_seed"]
    if not np.isfinite(n_grid) or int(n_grid) != int(n_example):
        raise RuntimeError(
            "Representative-grid consistency check failed: "
            f"heatmap N_seed={n_grid}, adaptive trajectory N_seed={n_example}. "
            "Decrease RK4_MAX_STEP or use solve_ivp for the relevant grid point."
        )
    print(
        "Representative-grid consistency check passed: "
        f"q={q}, Delta tau={delta_tau}, N_seed={int(n_grid)}."
    )

# -----------------------------------------------------------------------------
# 6. Plotting
# -----------------------------------------------------------------------------
def add_vector_field(ax, case, nbar_max=None, nx=21, ny=21):
    pb = phi_bar(case)
    if nbar_max is None:
        nbar_max = pb
    xg = np.linspace(0.02, 0.98, nx)
    ng = np.linspace(0.02, nbar_max - 0.02, ny)
    X, N = np.meshgrid(xg, ng)
    dX, dN = rhs_tau_array(case, X, N)
    speed = np.sqrt(dX*dX + dN*dN)
    dXn = np.divide(dX, speed, out=np.zeros_like(dX), where=speed > 0)
    dNn = np.divide(dN, speed, out=np.zeros_like(dN), where=speed > 0)
    ax.quiver(X, N, dXn, dNn, angles="xy", scale_units="xy", scale=32,
              width=0.003, color="black", alpha=0.32)


def plot_hybrid_phase_plane(ax, case, separatrix, result):
    pb = phi_bar(case)
    n_grid = np.linspace(0.0, pb, 600)
    x_sep = evaluate_separatrix(separatrix, n_grid)
    ax.fill_betweenx(n_grid, 0.0, x_sep, color=COL["host"], alpha=0.10)
    ax.fill_betweenx(n_grid, x_sep, 1.0, color=COL["cancer"], alpha=0.10)
    ax.plot(x_sep, n_grid, color="black", lw=2.4)
    add_vector_field(ax, case, pb)

    for segment in result["segments"]:
        ax.plot(segment["x"], segment["nbar"], color=COL["bistab"], lw=2.2, zorder=5)

    for wave, (before, after) in enumerate(zip(result["pulse_pre"], result["pulse_post"]), 1):
        ax.annotate("", xy=(after[0], after[1]), xytext=(before[0], before[1]),
                    arrowprops=dict(arrowstyle="-|>", color=COL["cancer"], lw=2.0,
                                    mutation_scale=11), zorder=7)
        ax.text(after[0] + 0.007, after[1] + 0.035, str(wave), fontsize=8,
                color=COL["cancer"], zorder=8)

    for eq in compute_equilibria(case):
        if eq["kind"] == "stable":
            ax.plot(eq["x"], eq["nbar"], "o", ms=8, mfc="black", mec="white", mew=1.2, zorder=9)
        elif eq["kind"] == "saddle":
            ax.plot(eq["x"], eq["nbar"], "*", ms=14, mfc="black", mec="black", zorder=9)

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.08, pb + 0.08)
    ax.set_xlabel(r"Cancer fraction $x$", fontsize=13)
    ax.set_ylabel(r"Conditioning $\bar n$", fontsize=13)
    ax.set_title(rf"Repeated subcritical waves: $N_{{\rm seed}}={result['n_seed']}$", fontsize=14)
    ax.tick_params(labelsize=10)


def plot_hybrid_time_series(ax, case, result):
    ax_n = ax.twinx()
    first = True
    for segment in result["segments"]:
        ax.plot(segment["tau"], segment["x"], color=COL["cancer"], lw=2.4,
                label=r"Cancer fraction $x$" if first else None)
        ax_n.plot(segment["tau"], segment["nbar"], color=COL["bistab"], lw=2.1,
                  label=r"Conditioning $\bar n$" if first else None)
        first = False

    for tau, before, after in zip(result["pulse_times"], result["pulse_pre"], result["pulse_post"]):
        ax.plot([tau, tau], [before[0], after[0]], color=COL["cancer"], lw=2.2)
        ax.axvline(tau, color="black", lw=0.8, alpha=0.16)

    ax.axhline(X_OUT, color="black", ls="--", lw=1.3, alpha=0.55)
    ax_n.axhline(1.0, color="black", ls=":", lw=1.3, alpha=0.60)
    ax.set_ylim(-0.03, 1.03)
    ax_n.set_ylim(-0.08, phi_bar(case) + 0.15)
    ax.set_xlabel(r"Clearance time $\tau$", fontsize=13)
    ax.set_ylabel(r"Cancer fraction $x$", fontsize=13, color=COL["cancer"])
    ax_n.set_ylabel(r"Conditioning $\bar n$", fontsize=13, color=COL["bistab"])
    ax.tick_params(axis="y", labelcolor=COL["cancer"], labelsize=10)
    ax_n.tick_params(axis="y", labelcolor=COL["bistab"], labelsize=10)
    ax.tick_params(axis="x", labelsize=10)
    ax.set_title(rf"$q={result['q']:.2f}$, $\Delta\tau={result['delta_tau']:.1f}$", fontsize=14)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_n.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)


def discrete_count_colormap(Z, max_waves):
    finite = Z[np.isfinite(Z)]
    max_count = int(np.max(finite)) if finite.size else max_waves
    colors = plt.cm.viridis(np.linspace(0.08, 0.95, max_count))
    cmap = ListedColormap(colors)
    cmap.set_bad("#D9D9D9")
    bounds = np.arange(0.5, max_count + 1.5, 1.0)
    norm = BoundaryNorm(bounds, cmap.N)
    return cmap, norm


def plot_number_heatmap(ax, grid):
    Z = np.ma.masked_invalid(grid["n_seed"])
    cmap, norm = discrete_count_colormap(grid["n_seed"], grid["max_waves"])
    mesh = ax.pcolormesh(grid["q"], grid["delta_tau"], Z,
                         cmap=cmap, norm=norm, shading="auto")
    cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(r"Number of waves $N_{\rm seed}$", fontsize=11)
    finite = grid["n_seed"][np.isfinite(grid["n_seed"])]
    max_count = int(np.max(finite)) if finite.size else grid["max_waves"]
    tick_values = np.unique(np.linspace(1, max_count, min(8, max_count), dtype=int))
    cbar.set_ticks(tick_values)
    cbar.ax.tick_params(labelsize=8)
    ax.set_xlabel(r"Wave size $q$", fontsize=13)
    ax.set_ylabel(r"Inter-wave interval $\Delta\tau$", fontsize=13)
    ax.set_title("Waves required for self-sustaining colonization", fontsize=14)
    ax.tick_params(labelsize=10)
    ax.text(0.03, 0.96, "grey: no crossing within finite pulse train",
            transform=ax.transAxes, ha="left", va="top", fontsize=8)


def plot_time_heatmap(ax, grid):
    #cmap = plt.cm.magma.copy()
    #cmap.set_bad("#D9D9D9")
    
    cmap = copy.copy(plt.get_cmap("viridis"))
    cmap.set_bad("#D9D9D9")
    
    mesh = ax.pcolormesh(grid["q"], grid["delta_tau"],
                         np.ma.masked_invalid(grid["tau_out"]),
                         cmap=cmap, shading="auto")
    cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label(r"Total time to outgrowth $T_{\rm out}$", fontsize=11)
    cbar.ax.tick_params(labelsize=8)
    ax.set_xlabel(r"Wave size $q$", fontsize=13)
    ax.set_ylabel(r"Inter-wave interval $\Delta\tau$", fontsize=13)
    ax.set_title(rf"Time to $x={grid['x_out']:.1f}$", fontsize=14)
    ax.tick_params(labelsize=10)


def plot_multiwave_figure(case, separatrix, example_result, grid,
                          filename="multiwave_priming"):
    fig, axs = plt.subplots(2, 2, figsize=(12.2, 10.2))
    plot_hybrid_phase_plane(axs[0, 0], case, separatrix, example_result)
    plot_hybrid_time_series(axs[0, 1], case, example_result)
    plot_number_heatmap(axs[1, 0], grid)
    plot_time_heatmap(axs[1, 1], grid)
    for label, ax in zip(("A", "B", "C", "D"), axs.ravel()):
        ax.text(-0.15, 1.08, label, transform=ax.transAxes,
                fontsize=18, fontweight="bold", va="top")
    fig.subplots_adjust(left=0.08, right=0.94, bottom=0.08, top=0.94,
                        wspace=0.34, hspace=0.30)
    png = os.path.join(OUTDIR, filename + ".png")
    svg = os.path.join(OUTDIR, filename + ".svg")
    fig.savefig(png, dpi=600, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.show()
    return png, svg


def plot_initial_condition_sensitivity(case, separatrix, sensitivity_grid,
                                       filename="multiwave_initial_condition_sensitivity"):
    fig, axs = plt.subplots(1, 2, figsize=(12.0, 4.8))

    # A: exact one-wave threshold.
    n_grid = np.linspace(0.0, phi_bar(case), 500)
    for x_pre in (0.0, 0.05, 0.10, 0.20):
        axs[0].plot(n_grid,
                    critical_wave_size_q(separatrix, x_pre, n_grid),
                    lw=2.2, label=rf"$x^-={x_pre:.2f}$")
    axs[0].axvline(1.0, color="black", ls=":", lw=1.3, alpha=0.6)
    axs[0].set_xlabel(r"Pre-wave conditioning $\bar n^-$", fontsize=13)
    axs[0].set_ylabel(r"Critical wave size $q_{\rm c}$", fontsize=13)
    axs[0].set_title("Exact one-wave basin-crossing threshold", fontsize=14)
    axs[0].legend(fontsize=9)
    axs[0].tick_params(labelsize=10)
    axs[0].set_ylim(-0.01, 0.36)

    # B: repeated-wave sensitivity to initial conditioning.
    Z = np.ma.masked_invalid(sensitivity_grid["n_seed"])
    cmap, norm = discrete_count_colormap(sensitivity_grid["n_seed"],
                                         sensitivity_grid["max_waves"])
    mesh = axs[1].pcolormesh(sensitivity_grid["q"],
                             sensitivity_grid["nbar0"], Z,
                             cmap=cmap, norm=norm, shading="auto")
    cbar = plt.colorbar(mesh, ax=axs[1], pad=0.02)
    cbar.set_label(r"Number of waves $N_{\rm seed}$", fontsize=11)
    finite = sensitivity_grid["n_seed"][np.isfinite(sensitivity_grid["n_seed"])]
    max_count = int(np.max(finite)) if finite.size else sensitivity_grid["max_waves"]
    tick_values = np.unique(np.linspace(1, max_count, min(8, max_count), dtype=int))
    cbar.set_ticks(tick_values)
    cbar.ax.tick_params(labelsize=8)
    axs[1].set_xlabel(r"Wave size $q$", fontsize=13)
    axs[1].set_ylabel(r"Initial conditioning $\bar n_0$", fontsize=13)
    axs[1].set_title(rf"Initial-state sensitivity at $\Delta\tau={sensitivity_grid['delta_tau']:.1f}$",
                     fontsize=14)
    axs[1].tick_params(labelsize=10)

    for label, ax in zip(("A", "B"), axs):
        ax.text(-0.14, 1.08, label, transform=ax.transAxes,
                fontsize=18, fontweight="bold", va="top")
    fig.subplots_adjust(left=0.08, right=0.95, bottom=0.16, top=0.88, wspace=0.30)
    png = os.path.join(OUTDIR, filename + ".png")
    svg = os.path.join(OUTDIR, filename + ".svg")
    fig.savefig(png, dpi=600, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    plt.show()
    return png, svg

# -----------------------------------------------------------------------------
# 7. Execute analysis
# -----------------------------------------------------------------------------
print("\n--- Impulsive multi-wave priming analysis ---")
print(f"phi_bar = {phi_bar(PRIMING_CASE):.6g}")
print(f"saddle = ({SEPARATRIX['saddle'][0]:.6f}, {SEPARATRIX['saddle'][1]:.6f})")
print(f"stable eigenvalue = {SEPARATRIX['stable_eigenvalue']:.6f}")
print(f"unprimed one-wave q_c = {float(critical_wave_size_q(SEPARATRIX, 0.0, 0.0)):.6f}")
print(f"unprimed one-wave eta_c = {float(critical_wave_size_eta(SEPARATRIX, 0.0, 0.0)):.6f}")
print(f"rapid-pulse N_seed for q={EXAMPLE_Q:.2f}: "
      f"{rapid_seeding_wave_count(SEPARATRIX, EXAMPLE_Q, EXAMPLE_X0, EXAMPLE_NBAR0)}")

EXAMPLE_RESULT = simulate_pulse_train(
    PRIMING_CASE, SEPARATRIX,
    q=EXAMPLE_Q, delta_tau=EXAMPLE_DELTA_TAU,
    x0=EXAMPLE_X0, nbar0=EXAMPLE_NBAR0,
    max_waves=MAX_WAVES, x_out=X_OUT,
    post_tau_max=POST_TAU_MAX, record=True,
)
print(f"representative train: N_seed={EXAMPLE_RESULT['n_seed']}, "
      f"T_out={EXAMPLE_RESULT['tau_out']:.6f} tau, "
      f"t_out={EXAMPLE_RESULT['t_out']:.6f}")

MULTIWAVE_GRID = compute_multiwave_grid_fast(
    PRIMING_CASE, SEPARATRIX,
    Q_VALUES, DELTA_TAU_VALUES,
    x0=0.0, nbar0=0.0,
    max_waves=MAX_WAVES, x_out=X_OUT,
)
check_representative_grid_consistency(MULTIWAVE_GRID, EXAMPLE_RESULT)

INITIAL_CONDITION_GRID = compute_initial_condition_grid_fast(
    PRIMING_CASE, SEPARATRIX,
    SENSITIVITY_Q_VALUES, SENSITIVITY_NBAR0_VALUES,
    delta_tau=SENSITIVITY_DELTA_TAU,
    x0=0.0, max_waves=MAX_WAVES,
)

MAIN_FIGURES = plot_multiwave_figure(
    PRIMING_CASE, SEPARATRIX, EXAMPLE_RESULT, MULTIWAVE_GRID
)
SENSITIVITY_FIGURES = plot_initial_condition_sensitivity(
    PRIMING_CASE, SEPARATRIX, INITIAL_CONDITION_GRID
)

# -----------------------------------------------------------------------------
# 8. Save reproducible data
# -----------------------------------------------------------------------------
np.savez_compressed(
    os.path.join(OUTDIR, "multiwave_priming_data.npz"),
    separatrix_x=SEPARATRIX["x"],
    separatrix_nbar=SEPARATRIX["nbar"],
    saddle=SEPARATRIX["saddle"],
    q_values=MULTIWAVE_GRID["q"],
    delta_tau_values=MULTIWAVE_GRID["delta_tau"],
    n_seed=MULTIWAVE_GRID["n_seed"],
    crossing_x=MULTIWAVE_GRID["crossing_x"],
    crossing_nbar=MULTIWAVE_GRID["crossing_nbar"],
    tau_post=MULTIWAVE_GRID["tau_post"],
    tau_out=MULTIWAVE_GRID["tau_out"],
    t_out=MULTIWAVE_GRID["t_out"],
    sensitivity_q=INITIAL_CONDITION_GRID["q"],
    sensitivity_nbar0=INITIAL_CONDITION_GRID["nbar0"],
    sensitivity_n_seed=INITIAL_CONDITION_GRID["n_seed"],
    sensitivity_delta_tau=INITIAL_CONDITION_GRID["delta_tau"],
    example_pulse_times=EXAMPLE_RESULT["pulse_times"],
    example_pulse_pre=EXAMPLE_RESULT["pulse_pre"],
    example_pulse_post=EXAMPLE_RESULT["pulse_post"],
)

with open(os.path.join(OUTDIR, "multiwave_representative_pulses.csv"),
          "w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow([
        "wave", "tau", "x_pre", "nbar_pre", "x_post", "nbar_post",
        "x_separatrix", "post_minus_separatrix"
    ])
    for wave, tau, pre, post in zip(
        range(1, len(EXAMPLE_RESULT["pulse_times"]) + 1),
        EXAMPLE_RESULT["pulse_times"],
        EXAMPLE_RESULT["pulse_pre"],
        EXAMPLE_RESULT["pulse_post"],
    ):
        xsep = float(evaluate_separatrix(SEPARATRIX, post[1]))
        writer.writerow([
            wave, tau, pre[0], pre[1], post[0], post[1],
            xsep, post[0] - xsep
        ])

print("Saved:")
for path in (*MAIN_FIGURES, *SENSITIVITY_FIGURES):
    print(" ", path)
print(" ", os.path.join(OUTDIR, "multiwave_priming_data.npz"))
print(" ", os.path.join(OUTDIR, "multiwave_representative_pulses.csv"))


### R2M4

In [ ]:
"""
R2M4 bifurcation, Hopf, and gate-sensitivity analysis.

WHERE TO USE:
- Append this code to Paper_corrections.ipynb, or run it as a standalone script
  from the project root.
- It writes the following files to ./Figures/:
    bifurcation_diagrams_endpoints.png/.svg/.npz
    gate_sensitivity.png/.svg/.npz
    gate_sensitivity_saddle_nodes.csv

This script is intentionally self-contained. It does not require the notebook's
Case or endpoint objects.
"""

from __future__ import annotations

from dataclasses import dataclass
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import brentq

# -----------------------------------------------------------------------------
# Parameters used in the submitted figures
# -----------------------------------------------------------------------------
ALPHA_PRIMING = 0.5
BETA_PRIMING = 0.6
GAMMA_PRIMING = 0.5
PHI_BAR_CLEARANCE = 0.7
PHI_BAR_PRIMING = 4.6

ALPHA_AWAKENING = 0.1
BETA_AWAKENING = 0.9
GAMMA_AWAKENING = 0.02
PHI_BAR_AWAKENING = 2.25
PHI_BAR_DORMANCY_DEMO = 0.35

NU_MAIN = 1.0
NU_VALUES = (1.0, 2.0, 4.0, 8.0)

OUTDIR = "Figures"
os.makedirs(OUTDIR, exist_ok=True)

COL = {
    "host": "#707071",
    "coexist": "#629FB7",
    "bistab": "#106E91",
    "cancer": "#AF4772",
}


# -----------------------------------------------------------------------------
# Endpoint definitions
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class GameEndpoints:
    alpha_h: float
    beta_h: float
    alpha_s: float
    beta_s: float


def endpoints_host_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """Low priming host-dominant endpoint to high priming cancer-dominant endpoint."""
    return GameEndpoints(alpha_h=-alpha, beta_h=beta, alpha_s=alpha, beta_s=-beta)


def endpoints_coexist_to_cancer(alpha: float, beta: float) -> GameEndpoints:
    """Low priming coexistence endpoint to high priming cancer-dominant endpoint."""
    return GameEndpoints(alpha_h=alpha, beta_h=beta, alpha_s=alpha, beta_s=-beta)


# -----------------------------------------------------------------------------
# General helpers
# -----------------------------------------------------------------------------
def hill_gate_general(nbar, nu):
    """f_nu(nbar) = nbar^nu/(1+nbar^nu), vectorized for nbar >= 0."""
    nbar = np.asarray(nbar, dtype=float)
    if np.any(nbar < 0):
        raise ValueError("nbar must be non-negative")
    z = nbar**nu
    return z / (1.0 + z)


def hill_gate_prime_general(nbar, nu):
    """Derivative of f_nu(nbar)."""
    nbar = np.asarray(nbar, dtype=float)
    if np.any(nbar < 0):
        raise ValueError("nbar must be non-negative")
    return nu * nbar ** (nu - 1.0) / (1.0 + nbar**nu) ** 2


def payoff_parameters(nbar, ends: GameEndpoints, nu):
    """Return alpha(nbar), beta(nbar), and f_nu(nbar)."""
    f = hill_gate_general(nbar, nu)
    a = (1.0 - f) * ends.alpha_h + f * ends.alpha_s
    b = (1.0 - f) * ends.beta_h + f * ends.beta_s
    return a, b, f


def jacobian_tau_parameters(x, nbar, ends: GameEndpoints, phi_bar_value, gamma, nu):
    """Jacobian of the tau = gamma*t system at an arbitrary state."""
    a, b, _ = payoff_parameters(float(nbar), ends, nu)
    a = float(a)
    b = float(b)

    U = x * (1.0 - x)
    Q = a - (a + b) * x

    J11 = ((1.0 - 2.0 * x) * Q - U * (a + b)) / gamma

    fp = float(hill_gate_prime_general(float(nbar), nu))
    a_n = fp * (ends.alpha_s - ends.alpha_h)
    b_n = fp * (ends.beta_s - ends.beta_h)
    J12 = U * (a_n - (a_n + b_n) * x) / gamma

    return np.array([[J11, J12], [phi_bar_value, -1.0]], dtype=float)


def stability_from_jacobian(J, tol=1e-9):
    """Classify a 2D equilibrium from its eigenvalues."""
    eig = np.linalg.eigvals(J)
    re = eig.real
    if np.all(re < -tol):
        return "stable"
    if re[0] * re[1] < -tol:
        return "saddle"
    if np.all(re > tol):
        return "unstable"
    return "non-hyperbolic"


def masked_by_stability(values, labels, target):
    """Return values with NaN outside the requested stability class."""
    values = np.asarray(values, dtype=float)
    labels = np.asarray(labels, dtype=object)
    return np.where(labels == target, values, np.nan)


# -----------------------------------------------------------------------------
# Exact equilibrium branches and saddle-node formula
# -----------------------------------------------------------------------------
def colonization_interior_branch(phi_bar_values):
    """Interior saddle E_s=(1/phi_bar,1), admissible only for phi_bar>1."""
    pb = np.asarray(phi_bar_values, dtype=float)
    x = np.full_like(pb, np.nan)
    mask = pb > 1.0
    x[mask] = 1.0 / pb[mask]
    return x


def awakening_parametric_branch(alpha, beta, nu, nbar_values):
    """
    Parametric interior equilibrium branch for the coexistence-to-cancer endpoint:

        x*(nbar) = alpha(1+nbar^nu) /
                   [(alpha+beta)+(alpha-beta)nbar^nu]
        phi_bar(nbar) = nbar/x*(nbar)

    The interval 0<nbar<=1 traces the biologically relevant low branch and,
    after the fold, the upper saddle branch ending at (phi_bar,x)=(1,1).
    """
    nbar = np.asarray(nbar_values, dtype=float)
    z = nbar**nu
    denom = (alpha + beta) + (alpha - beta) * z
    x = alpha * (1.0 + z) / denom
    phi_bar_values = nbar / x

    valid = (
        np.isfinite(x)
        & np.isfinite(phi_bar_values)
        & (x > 0.0)
        & (x <= 1.0 + 1e-10)
        & (phi_bar_values >= 0.0)
    )
    return phi_bar_values[valid], x[valid], nbar[valid]


def awakening_fold(alpha, beta, nu):
    """
    Biologically relevant saddle-node of the coexistence-to-cancer endpoint.

    With z=nbar^nu, folds satisfy
        (alpha-beta) z^2 + 2(alpha-beta*nu) z + (alpha+beta) = 0.
    """
    coeff = np.array(
        [alpha - beta, 2.0 * (alpha - beta * nu), alpha + beta],
        dtype=float,
    )
    roots = np.roots(coeff)
    candidates = []

    for root in roots:
        if abs(root.imag) > 1e-9 or root.real <= 0.0:
            continue
        z = float(root.real)
        nbar = z ** (1.0 / nu)
        denom = (alpha + beta) + (alpha - beta) * z
        if denom <= 0.0:
            continue
        x = alpha * (1.0 + z) / denom
        pb = nbar / x
        if 0.0 < x < 1.0 and 0.0 < nbar < 1.0 and pb > 1.0:
            candidates.append(
                {
                    "nu": float(nu),
                    "z_sn": z,
                    "nbar_sn": nbar,
                    "x_sn": x,
                    "phi_bar_sn": pb,
                }
            )

    if not candidates:
        raise RuntimeError(
            f"No admissible saddle-node found for alpha={alpha}, beta={beta}, nu={nu}."
        )

    # The relevant fold is the first loss of the low-conditioning branch.
    return min(candidates, key=lambda item: item["phi_bar_sn"])


def classify_awakening_branch(alpha, beta, gamma, nu, phi_values, x_values, nbar_values):
    """Evaluate stability, trace, determinant, and imaginary eigenvalue parts."""
    ends = endpoints_coexist_to_cancer(alpha=alpha, beta=beta)
    labels, traces, dets, max_imag = [], [], [], []

    for pb, x, nbar in zip(phi_values, x_values, nbar_values):
        J = jacobian_tau_parameters(x, nbar, ends, pb, gamma, nu)
        eig = np.linalg.eigvals(J)
        labels.append(stability_from_jacobian(J))
        traces.append(np.trace(J))
        dets.append(np.linalg.det(J))
        max_imag.append(np.max(np.abs(eig.imag)))

    return (
        np.asarray(labels, dtype=object),
        np.asarray(traces, dtype=float),
        np.asarray(dets, dtype=float),
        np.asarray(max_imag, dtype=float),
    )


def awakening_x_at_phi(alpha, beta, nu, phi_target, branch="stable"):
    """Interpolate x* on a requested branch at a specified environmental load."""
    nbar = np.linspace(1e-7, 1.0, 20000)
    pb, x, nb = awakening_parametric_branch(alpha, beta, nu, nbar)
    labels, _, _, _ = classify_awakening_branch(alpha, beta, GAMMA_AWAKENING, nu, pb, x, nb)
    mask = labels == branch
    pb_branch = pb[mask]
    x_branch = x[mask]
    order = np.argsort(pb_branch)
    pb_branch = pb_branch[order]
    x_branch = x_branch[order]
    if len(pb_branch) == 0 or not (pb_branch.min() <= phi_target <= pb_branch.max()):
        return np.nan
    return float(np.interp(phi_target, pb_branch, x_branch))


def count_admissible_awakening_roots(alpha, beta, nu, phi_bar):
    """Independent root-count check for interior equilibria at fixed phi_bar."""
    # Roots are found by solving phi_bar(nbar) - phi_bar = 0 over nbar in (0,1).
    n_grid = np.linspace(1e-6, 1.0 - 1e-6, 5000)
    pb_grid, _, nb_valid = awakening_parametric_branch(alpha, beta, nu, n_grid)
    # Need values aligned to valid nbar values.
    _, _, nb = awakening_parametric_branch(alpha, beta, nu, n_grid)
    vals = pb_grid - phi_bar
    roots = 0
    for a, b in zip(vals[:-1], vals[1:]):
        if a == 0 or a * b < 0:
            roots += 1
    return roots


# -----------------------------------------------------------------------------
# Numerical checks and table
# -----------------------------------------------------------------------------
def run_analytical_checks():
    print("\nSaddle-node sensitivity for alpha=0.1, beta=0.9")
    print("nu       z_SN       nbar_SN      x_SN      phi_bar_SN")
    rows = []
    for nu in NU_VALUES:
        fold = awakening_fold(ALPHA_AWAKENING, BETA_AWAKENING, nu)
        rows.append(fold)
        print(
            f"{nu:>3.0f}  {fold['z_sn']:10.6f}  {fold['nbar_sn']:10.6f}  "
            f"{fold['x_sn']:8.6f}  {fold['phi_bar_sn']:12.6f}"
        )

    x_low = awakening_x_at_phi(
        ALPHA_AWAKENING,
        BETA_AWAKENING,
        NU_MAIN,
        PHI_BAR_DORMANCY_DEMO,
        branch="stable",
    )
    print(
        f"\nStable interior equilibrium at phi_bar={PHI_BAR_DORMANCY_DEMO:.2f}: "
        f"x*={x_low:.6f}"
    )

    print("\nNo-Hopf numerical verification along interior branches")
    print("endpoint/nu        max(trace)      max|Im(lambda)|")

    ends_col = endpoints_host_to_cancer(ALPHA_PRIMING, BETA_PRIMING)
    pb_col = np.linspace(1.0001, 8.0, 2000)
    x_col = 1.0 / pb_col
    for nu in NU_VALUES:
        traces, imags = [], []
        for pb, x in zip(pb_col, x_col):
            J = jacobian_tau_parameters(x, 1.0, ends_col, pb, GAMMA_PRIMING, nu)
            eig = np.linalg.eigvals(J)
            traces.append(np.trace(J))
            imags.append(np.max(np.abs(eig.imag)))
        print(f"colonization/{nu:<4.0f}  {max(traces): .6e}    {max(imags): .6e}")

    nbar = np.linspace(1e-6, 1.0, 12000)
    for nu in NU_VALUES:
        pb, x, nb = awakening_parametric_branch(ALPHA_AWAKENING, BETA_AWAKENING, nu, nbar)
        _, traces, _, imags = classify_awakening_branch(
            ALPHA_AWAKENING, BETA_AWAKENING, GAMMA_AWAKENING, nu, pb, x, nb
        )
        print(f"awakening/{nu:<7.0f}  {np.max(traces): .6e}    {np.max(imags): .6e}")

    table = np.array(
        [[r["nu"], r["z_sn"], r["nbar_sn"], r["x_sn"], r["phi_bar_sn"]] for r in rows],
        dtype=float,
    )
    np.savetxt(
        os.path.join(OUTDIR, "gate_sensitivity_saddle_nodes.csv"),
        table,
        delimiter=",",
        header="nu,z_sn,nbar_sn,x_sn,phi_bar_sn",
        comments="",
    )
    return rows, x_low


# -----------------------------------------------------------------------------
# Main-text bifurcation figure
# -----------------------------------------------------------------------------
def plot_main_bifurcation_figure():
    stable_style = dict(lw=3.0, ls="-")
    saddle_style = dict(lw=2.5, ls="--")
    threshold_style = dict(color="black", lw=1.4, ls=":", alpha=0.7)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2))
    ax_a, ax_b = axes

    # Panel A: colonization/priming endpoint.
    pb_a = np.linspace(0.0, 5.2, 1500)
    mask_low = pb_a < 1.0
    mask_high = pb_a > 1.0

    ax_a.plot(pb_a, np.zeros_like(pb_a), color=COL["host"], label="stable", **stable_style)
    ax_a.plot(pb_a[mask_low], np.ones(np.count_nonzero(mask_low)), color="black", **saddle_style)
    ax_a.plot(pb_a[mask_high], np.ones(np.count_nonzero(mask_high)), color=COL["cancer"], **stable_style)

    pb_int = np.linspace(1.0001, 5.2, 1500)
    ax_a.plot(pb_int, 1.0 / pb_int, color="black", **saddle_style)

    ax_a.axvline(1.0, **threshold_style)
    ax_a.axvline(PHI_BAR_CLEARANCE, color=COL["host"], lw=1.4, ls="-.", alpha=0.8)
    ax_a.axvline(PHI_BAR_PRIMING, color=COL["bistab"], lw=1.4, ls="-.", alpha=0.8)
    ax_a.plot(1.0, 1.0, marker="o", ms=7, mfc="white", mec="black", mew=1.4, zorder=5)

    ax_a.text(1.03, 0.52, r"$\bar\phi=1$", rotation=90, va="center", fontsize=11)
    ax_a.text(PHI_BAR_CLEARANCE - 0.05, 0.08, "clearance", rotation=90,
              ha="right", va="bottom", fontsize=10, color=COL["host"])
    ax_a.text(PHI_BAR_PRIMING - 0.05, 0.08, "priming", rotation=90,
              ha="right", va="bottom", fontsize=10, color=COL["bistab"])

    ax_a.set_title("Colonization and priming endpoint", fontsize=16)
    ax_a.set_xlabel(r"Environmental load $\bar\phi$", fontsize=14)
    ax_a.set_ylabel(r"Equilibrium cancer fraction $x^*$", fontsize=14)
    ax_a.set_xlim(0.0, 5.2)
    ax_a.set_ylim(-0.04, 1.04)
    ax_a.tick_params(labelsize=11)

    # Panel B: dormancy-to-outgrowth endpoint, nu=1.
    nbar = np.linspace(1e-7, 1.0, 20000)
    pb_b, x_b, nb_b = awakening_parametric_branch(ALPHA_AWAKENING, BETA_AWAKENING, NU_MAIN, nbar)
    labels_b, traces_b, dets_b, imags_b = classify_awakening_branch(
        ALPHA_AWAKENING, BETA_AWAKENING, GAMMA_AWAKENING, NU_MAIN, pb_b, x_b, nb_b
    )
    fold = awakening_fold(ALPHA_AWAKENING, BETA_AWAKENING, NU_MAIN)

    pb_axis = np.linspace(0.0, 2.6, 1500)
    mask_low = pb_axis < 1.0
    mask_high = pb_axis > 1.0

    ax_b.plot(pb_axis, np.zeros_like(pb_axis), color="black", **saddle_style)
    ax_b.plot(pb_axis[mask_low], np.ones(np.count_nonzero(mask_low)), color="black", **saddle_style)
    ax_b.plot(pb_axis[mask_high], np.ones(np.count_nonzero(mask_high)), color=COL["cancer"], **stable_style)
    ax_b.plot(pb_b, masked_by_stability(x_b, labels_b, "stable"), color=COL["coexist"], **stable_style)
    ax_b.plot(pb_b, masked_by_stability(x_b, labels_b, "saddle"), color="black", **saddle_style)

    ax_b.axvline(1.0, **threshold_style)
    ax_b.axvline(fold["phi_bar_sn"], color=COL["coexist"], lw=1.6, ls=":", alpha=0.9)
    ax_b.axvline(PHI_BAR_AWAKENING, color=COL["cancer"], lw=1.4, ls="-.", alpha=0.8)

    ax_b.plot(fold["phi_bar_sn"], fold["x_sn"], marker="o", ms=7,
              mfc="white", mec=COL["coexist"], mew=1.6, zorder=6)
    ax_b.plot(1.0, 1.0, marker="o", ms=7, mfc="white", mec="black", mew=1.4, zorder=5)

    x_demo = awakening_x_at_phi(ALPHA_AWAKENING, BETA_AWAKENING, NU_MAIN,
                                PHI_BAR_DORMANCY_DEMO, branch="stable")
    ax_b.plot(PHI_BAR_DORMANCY_DEMO, x_demo, marker="o", ms=6, color=COL["coexist"], zorder=7)

    ax_b.text(1.03, 0.54, r"$\bar\phi=1$", rotation=90, va="center", fontsize=11)
    ax_b.text(fold["phi_bar_sn"] + 0.035, 0.52, r"$\bar\phi_{\mathrm{SN}}=2$",
              rotation=90, va="center", fontsize=11, color=COL["coexist"])
    ax_b.text(PHI_BAR_AWAKENING + 0.03, 0.07, "awakening\nexample",
              rotation=90, va="bottom", fontsize=10, color=COL["cancer"])
    ax_b.annotate("low-load\ncoexistence", xy=(PHI_BAR_DORMANCY_DEMO, x_demo), xytext=(0.62, 0.27),
                  arrowprops=dict(arrowstyle="->", lw=1.0, color=COL["coexist"]),
                  fontsize=10, color=COL["coexist"])

    ax_b.set_title("Dormancy-to-outgrowth endpoint", fontsize=16)
    ax_b.set_xlabel(r"Environmental load $\bar\phi$", fontsize=14)
    ax_b.set_ylabel(r"Equilibrium cancer fraction $x^*$", fontsize=14)
    ax_b.set_xlim(0.0, 2.6)
    ax_b.set_ylim(-0.04, 1.04)
    ax_b.tick_params(labelsize=11)

    handles = [
        Line2D([0], [0], color=COL["coexist"], lw=3.0, ls="-", label="stable interior"),
        Line2D([0], [0], color=COL["host"], lw=3.0, ls="-", label="stable host boundary"),
        Line2D([0], [0], color=COL["cancer"], lw=3.0, ls="-", label="stable cancer boundary"),
        Line2D([0], [0], color="black", lw=2.5, ls="--", label="saddle branch"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor="white",
               markeredgecolor="black", markersize=7, label="non-hyperbolic point"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=True,
               fontsize=10, bbox_to_anchor=(0.5, -0.01))

    for label, ax in zip(("A", "B"), axes):
        ax.text(-0.13, 1.04, label, transform=ax.transAxes,
                fontsize=17, fontweight="bold", va="top")

    fig.subplots_adjust(left=0.09, right=0.98, top=0.88, bottom=0.22, wspace=0.27)
    fig.savefig(os.path.join(OUTDIR, "bifurcation_diagrams_endpoints.svg"))
    fig.savefig(os.path.join(OUTDIR, "bifurcation_diagrams_endpoints.png"), dpi=600)

    np.savez(
        os.path.join(OUTDIR, "bifurcation_diagrams_endpoints.npz"),
        colonization_phi=pb_int,
        colonization_saddle_x=1.0 / pb_int,
        awakening_phi=pb_b,
        awakening_x=x_b,
        awakening_nbar=nb_b,
        awakening_trace=traces_b,
        awakening_det=dets_b,
        awakening_max_imag=imags_b,
        awakening_stability=labels_b.astype(str),
    )
    return fig


# -----------------------------------------------------------------------------
# Supplementary gate-sensitivity figure
# -----------------------------------------------------------------------------
def plot_gate_sensitivity_figure():
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.6))
    ax_a, ax_b = axes

    nbar = np.linspace(0.0, 2.0, 1000)
    for nu in NU_VALUES:
        ax_a.plot(nbar, hill_gate_general(nbar, nu), lw=2.5, label=rf"$\nu={nu:g}$")
    ax_a.axvline(1.0, color="black", lw=1.2, ls=":", alpha=0.7)
    ax_a.axhline(0.5, color="black", lw=1.2, ls=":", alpha=0.7)
    ax_a.set_xlabel(r"Conditioning factor level $\bar n$", fontsize=14)
    ax_a.set_ylabel(r"Tolerance gate $f_\nu(\bar n)$", fontsize=14)
    ax_a.set_xlim(0.0, 2.0)
    ax_a.set_ylim(-0.02, 1.02)
    ax_a.set_title("Gate sharpness", fontsize=16)
    ax_a.tick_params(labelsize=11)
    ax_a.legend(frameon=True, fontsize=10)

    nu_grid = np.linspace(1.0, 10.0, 400)
    phi_sn = np.array([awakening_fold(ALPHA_AWAKENING, BETA_AWAKENING, nu)["phi_bar_sn"] for nu in nu_grid])
    ax_b.plot(nu_grid, phi_sn, lw=3.0, color=COL["coexist"])

    point_rows = [awakening_fold(ALPHA_AWAKENING, BETA_AWAKENING, nu) for nu in NU_VALUES]
    ax_b.scatter([r["nu"] for r in point_rows], [r["phi_bar_sn"] for r in point_rows],
                 s=45, facecolor="white", edgecolor=COL["coexist"], linewidth=1.5, zorder=4)
    for row in point_rows:
        ax_b.annotate(f"{row['phi_bar_sn']:.2f}", (row["nu"], row["phi_bar_sn"]),
                      xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9)

    ax_b.axhline(PHI_BAR_AWAKENING, color=COL["cancer"], lw=1.5, ls="-.", alpha=0.85,
                 label=rf"awakening example $\bar\phi={PHI_BAR_AWAKENING}$")
    nu_cross = brentq(
        lambda nu: awakening_fold(ALPHA_AWAKENING, BETA_AWAKENING, nu)["phi_bar_sn"] - PHI_BAR_AWAKENING,
        1.0,
        2.0,
    )
    ax_b.axvline(nu_cross, color="black", lw=1.2, ls=":", alpha=0.7)
    ax_b.plot(nu_cross, PHI_BAR_AWAKENING, marker="o", ms=6,
              mfc="white", mec="black", mew=1.2, zorder=5)
    ax_b.annotate(rf"$\nu_c\simeq{nu_cross:.2f}$", xy=(nu_cross, PHI_BAR_AWAKENING),
                  xytext=(2.0, 2.65), arrowprops=dict(arrowstyle="->", lw=1.0, color="black"),
                  fontsize=10)

    ax_b.set_xlabel(r"Gate exponent $\nu$", fontsize=14)
    ax_b.set_ylabel(r"Saddle-node load $\bar\phi_{\mathrm{SN}}$", fontsize=14)
    ax_b.set_xlim(1.0, 10.0)
    ax_b.set_ylim(1.0, max(phi_sn) * 1.08)
    ax_b.set_title("Sensitivity of the awakening threshold", fontsize=16)
    ax_b.tick_params(labelsize=11)
    ax_b.legend(frameon=True, fontsize=9, loc="lower right")

    for label, ax in zip(("A", "B"), axes):
        ax.text(-0.13, 1.04, label, transform=ax.transAxes,
                fontsize=17, fontweight="bold", va="top")

    fig.subplots_adjust(left=0.10, right=0.98, top=0.87, bottom=0.16, wspace=0.27)
    fig.savefig(os.path.join(OUTDIR, "gate_sensitivity.svg"))
    fig.savefig(os.path.join(OUTDIR, "gate_sensitivity.png"), dpi=600)

    np.savez(
        os.path.join(OUTDIR, "gate_sensitivity.npz"),
        nbar=nbar,
        nu_values=np.asarray(NU_VALUES),
        gate_values=np.vstack([hill_gate_general(nbar, nu) for nu in NU_VALUES]),
        nu_grid=nu_grid,
        phi_bar_sn=phi_sn,
        nu_cross=nu_cross,
    )
    print(f"\nAt phi_bar={PHI_BAR_AWAKENING}, the fold is crossed at nu={nu_cross:.6f}.")
    return fig, nu_cross


if __name__ == "__main__":
    fold_rows, x_low = run_analytical_checks()
    plot_main_bifurcation_figure()
    plot_gate_sensitivity_figure()
    plt.show()

### R2M6 - Corrected figs 4 and 5

In [ ]:
"""
R2M6 final patch: unified endpoint parameterization for Figures 4 and 5.

Reviewer issue
--------------
The original Figure 4/5 examples changed endpoint family, interaction magnitudes
(alpha, beta), and environmental load at the same time. This patch regenerates
both the Figure 4 time-course examples and the matching Figure 5 phase portraits
so that each endpoint family uses one fixed interaction parameterization:

    Colonization / priming endpoint:        alpha=0.5, beta=0.6
        Fig. 4A / 5A clearance:             phi-bar = 0.7
        Fig. 4D / 5D priming threshold:     phi-bar = 4.6

    Dormancy-to-outgrowth endpoint:         alpha=0.1, beta=0.9
        Fig. 4B / 5B stable dormancy:       phi-bar = 0.35
        Fig. 4C / 5C awakening:             phi-bar = 2.25

Where to run
------------
Run this cell/file in Paper_corrections.ipynb AFTER the cells defining:
    Case, endpoints_host_to_cancer, endpoints_coexist_to_cancer,
    run_case, plot_case_timeseries, plot_fig4_phase_portraits.

Important naming note
---------------------
The notebook function `plot_fig4_phase_portraits` has a legacy name, but it
plots the phase portraits used as manuscript Figure 5. This patch calls that
function with filename_base="fig5_phase_portraits" and writes/copies the output
as Figure 5 files.

Outputs
-------
Corrected Figure 4 time-course panels:
    Figures/fig3_clearance_stack_t.png/.svg
    Figures/fig3_dormancy_stack_t.png/.svg
    Figures/fig3_awakening_stack_t.png/.svg
    Figures/fig3_primed_threshold_stack_t.png/.svg

Clear aliases for Figure 4 panels:
    Figures/Fig4A_clearance_timeseries.png/.svg
    Figures/Fig4B_dormancy_timeseries.png/.svg
    Figures/Fig4C_awakening_timeseries.png/.svg
    Figures/Fig4D_primed_threshold_timeseries.png/.svg

Corrected Figure 5 phase portraits:
    Figures/fig5_phase_portraits.png/.svg
    Figures/fig5_phase_portraits_clearance.png/.svg
    Figures/fig5_phase_portraits_dormancy.png/.svg
    Figures/fig5_phase_portraits_awakening.png/.svg
    Figures/fig5_phase_portraits_primed_threshold.png/.svg

Manuscript composite files:
    Figures/Fig4.png
    Figures/Fig5.png

Existing Figures/Fig4.png and Figures/Fig5.png are backed up before overwriting.
"""

from __future__ import annotations

import math
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg


# Set to False in order to inspect Fig4_R2M6.png/Fig5_R2M6.png before
# replacing the manuscript files Figures/Fig4.png and Figures/Fig5.png.
OVERWRITE_MANUSCRIPT_FIGURES = True
OUTDIR = "Figures"

REQUIRED_NAMES = [
    "Case",
    "endpoints_host_to_cancer",
    "endpoints_coexist_to_cancer",
    "run_case",
    "plot_case_timeseries",
    "plot_fig4_phase_portraits",  # legacy function name; generates manuscript Fig. 5
]

missing = [name for name in REQUIRED_NAMES if name not in globals()]
if missing:
    raise RuntimeError(
        "Run this patch after the notebook cells defining the model and plotting "
        f"functions. Missing names: {missing}"
    )


FIG4_PANEL_ORDER = ["clearance", "dormancy", "awakening", "primed_threshold"]
FIG4_PANEL_ALIASES = {
    "clearance": "Fig4A_clearance_timeseries",
    "dormancy": "Fig4B_dormancy_timeseries",
    "awakening": "Fig4C_awakening_timeseries",
    "primed_threshold": "Fig4D_primed_threshold_timeseries",
}
FIG5_PANEL_ALIASES = {
    "clearance": "Fig5A_clearance_phase_portrait",
    "dormancy": "Fig5B_dormancy_phase_portrait",
    "awakening": "Fig5C_awakening_phase_portrait",
    "primed_threshold": "Fig5D_primed_threshold_phase_portrait",
}


def build_unified_endpoint_cases():
    """Corrected case builder for the Reviewer 2 Figure 4/5 parameter audit."""
    cases = {}

    # A) Clearance: same colonization endpoint as priming panel D.
    # phi-bar = phi/(gamma*k) = 0.7/(0.5*2.0) = 0.7
    cases["clearance"] = Case(
        name="Clearance",
        ends=endpoints_host_to_cancer(alpha=0.5, beta=0.6),
        phi=0.7,
        gamma=0.5,
        k=2.0,
        nu=1.0,
        tmax=80.0,
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0,
    )

    # B) Stable dormancy: same dormancy-to-outgrowth endpoint as awakening panel C.
    # phi-bar = 0.7/(1.0*2.0) = 0.35
    cases["dormancy"] = Case(
        name="Stable dormancy",
        ends=endpoints_coexist_to_cancer(alpha=0.1, beta=0.9),
        phi=0.7,
        gamma=1.0,
        k=2.0,
        nu=1.0,
        tmax=80.0,  # longer than the old 50 to make convergence easier to see
        x0s=(0.02, 0.10, 0.80, 0.98),
        n0=0.0,
    )

    # C) Awakening: unchanged; same dormancy-to-outgrowth endpoint as panel B.
    # phi-bar = 0.09/(0.02*2.0) = 2.25
    cases["awakening"] = Case(
        name="Awakening",
        ends=endpoints_coexist_to_cancer(alpha=0.1, beta=0.9),
        phi=0.09,
        gamma=0.02,
        k=2.0,
        nu=1.0,
        tmax=720.0,
        x0s=(0.02, 0.10, 0.40, 0.80),
        n0=0.0,
    )

    # D) Priming and multi-wave seeding: unchanged; same colonization endpoint as panel A.
    # phi-bar = 0.23/(0.5*0.1) = 4.6; nbar0=n0/k=1
    cases["primed_threshold"] = Case(
        name="Primed niche threshold",
        ends=endpoints_host_to_cancer(alpha=0.5, beta=0.6),
        phi=0.23,
        gamma=0.5,
        k=0.1,
        nu=1.0,
        tmax=50.0,
        x0s=(0.20, 0.40),
        n0=0.1,
    )

    return cases


def phibar(case):
    """Environmental load phi-bar = phi/(gamma*k)."""
    return case.phi / (case.gamma * case.k)


def assert_unified_endpoint_cases(cases):
    """Fail loudly if the corrected endpoint grouping is broken."""
    a = cases["clearance"].ends
    d = cases["primed_threshold"].ends
    b = cases["dormancy"].ends
    c = cases["awakening"].ends

    assert a == d, "Panels A and D must share the same colonization/priming endpoint."
    assert b == c, "Panels B and C must share the same dormancy-to-outgrowth endpoint."

    expected_phibar = {
        "clearance": 0.7,
        "dormancy": 0.35,
        "awakening": 2.25,
        "primed_threshold": 4.6,
    }
    for key, expected in expected_phibar.items():
        got = phibar(cases[key])
        assert math.isclose(got, expected, rel_tol=1e-12, abs_tol=1e-12), (
            f"Unexpected phi-bar for {key}: got {got}, expected {expected}"
        )


def print_unified_case_summary(cases):
    print("\nCorrected unified-endpoint cases")
    print("key                  endpoint (alpha0,beta0)->(alpha1,beta1)       phi-bar")
    print("-" * 86)
    for key in ["clearance", "primed_threshold", "dormancy", "awakening"]:
        case = cases[key]
        e = case.ends
        print(
            f"{key:21s} ({e.alpha_h:+.3f},{e.beta_h:+.3f})"
            f" -> ({e.alpha_s:+.3f},{e.beta_s:+.3f})        {phibar(case):.3f}"
        )


def backup_file(path: Path):
    """Create a stable backup once before overwriting a manuscript figure."""
    if not path.exists():
        return None
    backup = path.with_name(path.stem + "_pre_R2M6_backup" + path.suffix)
    if not backup.exists():
        shutil.copy2(path, backup)
    return backup


def copy_existing(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(f"Expected output file was not created: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def alias_fig4_timecourse_panels(outdir: str = OUTDIR):
    """Copy notebook-generated time-course panels to clear Fig4A-D aliases."""
    out = Path(outdir)
    for key, alias in FIG4_PANEL_ALIASES.items():
        for ext in ("png", "svg"):
            src = out / f"fig3_{key}_stack_t.{ext}"
            dst = out / f"{alias}.{ext}"
            copy_existing(src, dst)


def alias_fig5_phase_portrait_panels(outdir: str = OUTDIR):
    """Copy notebook-generated phase-portrait panels to clear Fig5A-D aliases."""
    out = Path(outdir)
    for key, alias in FIG5_PANEL_ALIASES.items():
        for ext in ("png", "svg"):
            src = out / f"fig5_phase_portraits_{key}.{ext}"
            dst = out / f"{alias}.{ext}"
            copy_existing(src, dst)


def assemble_fig4_timecourse_composite(outdir: str = OUTDIR, output_name: str = "Fig4_R2M6.png"):
    """Assemble corrected Fig. 4 time-course panels into a 2x2 manuscript-style PNG."""
    out = Path(outdir)
    panel_paths = [out / f"{FIG4_PANEL_ALIASES[key]}.png" for key in FIG4_PANEL_ORDER]
    for path in panel_paths:
        if not path.exists():
            raise FileNotFoundError(f"Missing Fig. 4 panel for assembly: {path}")

    fig, axes = plt.subplots(2, 2, figsize=(13.0, 12.0), constrained_layout=True)
    for ax, path, letter in zip(axes.ravel(), panel_paths, ["A", "B", "C", "D"]):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.text(
            0.015,
            0.985,
            letter,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=22,
            fontweight="bold",
            color="black",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2),
        )

    output = out / output_name
    fig.savefig(output, dpi=600, bbox_inches="tight")
    plt.close(fig)
    return output


def regenerate_unified_endpoint_figures(outdir: str = OUTDIR):
    """Regenerate corrected Figure 4 and Figure 5 outputs."""
    Path(outdir).mkdir(parents=True, exist_ok=True)

    cases = build_unified_endpoint_cases()
    assert_unified_endpoint_cases(cases)
    print_unified_case_summary(cases)

    # -------------------------
    # Figure 4: time-course panels
    # -------------------------
    all_outs = {}
    for key in FIG4_PANEL_ORDER:
        outs = run_case(cases[key], npts=2000)
        all_outs[key] = outs
        plot_case_timeseries(
            key,
            cases[key],
            outs,
            outdir=outdir,
            layout="stack",
            time_axis="t",
            label_x0=(key == "primed_threshold"),
        )

    alias_fig4_timecourse_panels(outdir)
    fig4_r2m6 = assemble_fig4_timecourse_composite(outdir, output_name="Fig4_R2M6.png")

    # -------------------------
    # Figure 5: phase portraits
    # -------------------------
    # Legacy function name in the notebook, but this is the manuscript Fig. 5.
    plot_fig4_phase_portraits(
        cases,
        outdir=outdir,
        filename_base="fig5_phase_portraits",
        include_basin_items=True,
        save_individual=True,
    )
    alias_fig5_phase_portrait_panels(outdir)

    fig5_r2m6 = Path(outdir) / "Fig5_R2M6.png"
    copy_existing(Path(outdir) / "fig5_phase_portraits.png", fig5_r2m6)
    copy_existing(Path(outdir) / "fig5_phase_portraits.svg", Path(outdir) / "Fig5_R2M6.svg")

    # Optional but enabled by default: update the files used by rev_manuscript.tex.
    if OVERWRITE_MANUSCRIPT_FIGURES:
        fig4_target = Path(outdir) / "Fig4.png"
        fig5_target = Path(outdir) / "Fig5.png"
        backup_file(fig4_target)
        backup_file(fig5_target)
        copy_existing(fig4_r2m6, fig4_target)
        copy_existing(Path(outdir) / "fig5_phase_portraits.png", fig5_target)
        print("\nUpdated manuscript figure files:")
        print(f"  {fig4_target}")
        print(f"  {fig5_target}")
        print("Backups, if originals existed, were written as *_pre_R2M6_backup.*")
    else:
        print("\nOVERWRITE_MANUSCRIPT_FIGURES=False, so manuscript Fig4.png/Fig5.png were not replaced.")
        print(f"Inspect corrected outputs: {fig4_r2m6} and {fig5_r2m6}")

    return cases, all_outs


# Execute immediately when pasted/run after the required notebook cells.
cases, all_outs = regenerate_unified_endpoint_figures(outdir=OUTDIR)

### R2M3

In [ ]:
"""
R2M3 low-burden dormancy checks.

Purpose
-------
Numerically verify the representative low-burden fixed point used in
Figs. 4B/5B and generate the small robustness table suggested for the
supplementary R2M3 response.

This code is specific to the dormancy-to-outgrowth endpoint with
(alpha_0, beta_0) = (alpha, beta), (alpha_1, beta_1) = (alpha, -beta),
and the nu=1 tolerance gate, for which beta(nbar)=beta*(1-nbar)/(1+nbar).
Do not reuse the closed-form beta(nbar) outside this endpoint/gate case.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable, Optional

import numpy as np
from scipy.optimize import brentq


@dataclass(frozen=True)
class LowBurdenResult:
    alpha: float
    beta: float
    phibar: float
    x_star: float
    nbar_star: float
    stability_lhs: float
    stability_rhs: float
    stable: bool
    phibar_sn: Optional[float]


def beta_eff_nu1_dormancy_endpoint(x: float, alpha: float, beta: float, phibar: float) -> float:
    """Effective beta at equilibrium nbar=phibar*x for the nu=1 dormancy endpoint."""
    nbar = phibar * x
    return beta * (1.0 - nbar) / (1.0 + nbar)


def q_equilibrium(x: float, alpha: float, beta: float, phibar: float) -> float:
    """Interior x-nullcline equation Q(x)=alpha-(alpha+beta_eff)x."""
    b_eff = beta_eff_nu1_dormancy_endpoint(x, alpha, beta, phibar)
    return alpha - (alpha + b_eff) * x


def find_low_burden_root(alpha: float, beta: float, phibar: float) -> float:
    """Find the admissible low-burden interior root in (0,1)."""
    xs = np.linspace(1e-10, 1.0 - 1e-10, 20_000)
    vals = np.array([q_equilibrium(x, alpha, beta, phibar) for x in xs])
    roots = []
    for i in range(len(xs) - 1):
        if vals[i] == 0:
            roots.append(xs[i])
        elif vals[i] * vals[i + 1] < 0:
            roots.append(brentq(q_equilibrium, xs[i], xs[i + 1], args=(alpha, beta, phibar)))
    if not roots:
        raise RuntimeError(f"No interior root found for alpha={alpha}, beta={beta}, phibar={phibar}")
    return min(roots)


def stability_terms(alpha: float, beta: float, phibar: float, x_star: float) -> tuple[float, float, bool]:
    """
    Supplementary stability condition for the nu=1 dormancy endpoint:
    alpha/beta > 2*phibar*x_star^2/(1+phibar*x_star)^2.
    """
    lhs = alpha / beta
    rhs = 2.0 * phibar * x_star**2 / (1.0 + phibar * x_star) ** 2
    return lhs, rhs, lhs > rhs


def saddle_node_load_nu1(alpha: float, beta: float) -> Optional[float]:
    """Closed-form nu=1 saddle-node load for the dormancy-to-outgrowth endpoint."""
    if not (alpha > 0 and beta > alpha):
        return None
    return (3.0 * beta - alpha - 2.0 * math.sqrt(2.0 * beta * (beta - alpha))) / alpha


def evaluate(alpha: float, beta: float, phibar: float = 0.35) -> LowBurdenResult:
    x_star = find_low_burden_root(alpha, beta, phibar)
    nbar_star = phibar * x_star
    lhs, rhs, stable = stability_terms(alpha, beta, phibar, x_star)
    return LowBurdenResult(
        alpha=alpha,
        beta=beta,
        phibar=phibar,
        x_star=x_star,
        nbar_star=nbar_star,
        stability_lhs=lhs,
        stability_rhs=rhs,
        stable=stable,
        phibar_sn=saddle_node_load_nu1(alpha, beta),
    )


def print_results(results: Iterable[LowBurdenResult]) -> None:
    print("alpha beta phibar x_star nbar_star alpha/beta stability_rhs stable phibar_SN")
    for r in results:
        phisn = "NA" if r.phibar_sn is None else f"{r.phibar_sn:.6f}"
        print(
            f"{r.alpha:.2f} {r.beta:.2f} {r.phibar:.2f} "
            f"{r.x_star:.6f} {r.nbar_star:.6f} "
            f"{r.stability_lhs:.6f} {r.stability_rhs:.6f} "
            f"{r.stable} {phisn}"
        )


if __name__ == "__main__":
    # Representative Fig. 4B/5B parameter set.
    representative = evaluate(alpha=0.10, beta=0.90, phibar=0.35)
    print("Representative low-burden dormancy fixed point")
    print_results([representative])

    print("\nRobustness table: decreasing alpha/beta lowers x* while preserving the saddle-node")
    robustness = [evaluate(alpha=a, beta=b, phibar=0.35) for a, b in [(0.10, 0.90), (0.05, 0.95), (0.02, 0.98)]]
    print_results(robustness)